In [0]:
%scala
val basePath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/"

val folders = Seq(
  "batch/bronze/customers",
  "batch/bronze/products",
  "batch/bronze/orders",
  "batch/silver/customers",
  "batch/silver/products",
  "batch/silver/orders",
  "batch/rejected/customers",
  "batch/rejected/products",
  "batch/rejected/orders",
  "batch/gold/dim_customer",
  "batch/gold/dim_product",
  "batch/gold/dim_date",
  "batch/gold/fact_orders",
  "streaming",
  "checkpoints",
  "logs"
)

folders.foreach { folder =>
  dbutils.fs.mkdirs(basePath + folder)
}

println("PROJECT NEXUS folder structure created!")

PROJECT NEXUS folder structure created!
val basePath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/
val folders: Seq[String] = List(batch/bronze/customers, batch/bronze/products, batch/bronze/orders, batch/silver/customers, batch/silver/products, batch/silver/orders, batch/rejected/customers, batch/rejected/products, batch/rejected/orders, batch/gold/dim_customer, batch/gold/dim_product, batch/gold/dim_date, batch/gold/fact_orders, streaming, checkpoints, logs)

In [0]:
%scala
val storagePath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/"

display(dbutils.fs.ls(storagePath))

path,name,size,modificationTime
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/,batch/,0,1795194243000
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/checkpoints/,checkpoints/,0,1795194243000
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/logs/,logs/,0,1795194243000
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/streaming/,streaming/,0,1795194243000


In [0]:
%scala
val basePath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/"

display(dbutils.fs.ls(basePath + "customers"))


path,name,size,modificationTime
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,customers.csv,2757,1795194520000


In [0]:
%scala
display(dbutils.fs.ls(basePath + "products"))


path,name,size,modificationTime
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,products.csv,1068,1795194578000


In [0]:
%scala
display(dbutils.fs.ls(basePath + "orders"))

path,name,size,modificationTime
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,orders.csv,10897,1795194544000


In [0]:
%scala
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._
import org.apache.spark.sql.DataFrame

import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._
import org.apache.spark.sql.DataFrame

In [0]:
%scala
val basePath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/"

val customersSource = basePath + "bronze/customers/customers.csv"
val productsSource  = basePath + "bronze/products/products.csv"
val ordersSource    = basePath + "bronze/orders/orders.csv"

val customersTarget = basePath + "bronze_delta/customers"
val productsTarget  = basePath + "bronze_delta/products"
val ordersTarget    = basePath + "bronze_delta/orders"

val basePath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/
val customersSource: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv
val productsSource: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv
val ordersSource: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv
val customersTarget: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze_delta/customers
val productsTarget: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze_delta/products
val ordersTarget: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze_delta/orders

In [0]:
%scala
val customerSchema = StructType(Seq(
  StructField("customer_id", StringType, true),
  StructField("customer_name", StringType, true),
  StructField("email", StringType, true),
  StructField("city", StringType, true),
  StructField("signup_date", StringType, true)
))

val productSchema = StructType(Seq(
  StructField("product_id", StringType, true),
  StructField("product_name", StringType, true),
  StructField("category", StringType, true),
  StructField("unit_price", StringType, true),
  StructField("active", StringType, true)
))

val orderSchema = StructType(Seq(
  StructField("order_id", StringType, true),
  StructField("customer_id", StringType, true),
  StructField("product_id", StringType, true),
  StructField("quantity", StringType, true),
  StructField("order_date", StringType, true),
  StructField("order_status", StringType, true),
  StructField("last_updated", StringType, true)
))

val customerSchema: org.apache.spark.sql.types.StructType = StructType(StructField(customer_id,StringType,true),StructField(customer_name,StringType,true),StructField(email,StringType,true),StructField(city,StringType,true),StructField(signup_date,StringType,true))
val productSchema: org.apache.spark.sql.types.StructType = StructType(StructField(product_id,StringType,true),StructField(product_name,StringType,true),StructField(category,StringType,true),StructField(unit_price,StringType,true),StructField(active,StringType,true))
val orderSchema: org.apache.spark.sql.types.StructType = StructType(StructField(order_id,StringType,true),StructField(customer_id,StringType,true),StructField(product_id,StringType,true),StructField(quantity,StringType,true),StructField(order_date,StringType,true),StructField(order_status,StringType,true),StructField(last_updated,StringType,true))

In [0]:
%scala
def readSource(
    path: String,
    schema: StructType
): DataFrame = {

  spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .schema(schema)
    .load(path)
    .withColumn("_source_file", input_file_name())
    .withColumn("_ingestion_timestamp", current_timestamp())
}

def readSource(path: String, schema: org.apache.spark.sql.types.StructType): org.apache.spark.sql.DataFrame

In [0]:
%scala
val customersDF =
  readSource(customersSource, customerSchema)

val productsDF =
  readSource(productsSource, productSchema)

val ordersDF =
  readSource(ordersSource, orderSchema)

val customersDF: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 5 more fields]
val productsDF: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 5 more fields]
val ordersDF: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 7 more fields]

In [0]:
%scala
customersDF.show(10, false)

productsDF.show(10, false)

ordersDF.show(10, false)

+-----------+-------------+-----------------------------+---------+-----------+-------------------------------------------------------------------------------------------------------+-------------------------+
customer_id|customer_name|email |city |signup_date|_source_file |_ingestion_timestamp |
+-----------+-------------+-----------------------------+---------+-----------+-------------------------------------------------------------------------------------------------------+-------------------------+
CUST001 |Priya Patel |priya.patel1@mailexample.in |Mumbai |2025-06-24 |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv|2026-11-20 17:20:56.20842|
CUST002 |Sneha Gupta |sneha.gupta2@mailexample.in |Kolkata |2025-03-13 |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv|2026-11-20 17:20:56.20842|
CUST003 |Divya Singh |divya.singh3@mailexample.in |Kochi |2025-02-08 |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv|2026-11-20 17:20:56.20842|
CUST004 |Suresh Gupta |suresh.gupta4@mailexample.in |Kochi |2023-09-03 |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv|2026-11-20 17:20:56.20842|
CUST005 |Aarav Gupta |aarav.gupta5@mailexample.in |Mumbai |2023-04-24 |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv|2026-11-20 17:20:56.20842|
CUST006 |Vikram Sharma|vikram.sharma6@mailexample.in|Kochi |2023-01-12 |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv|2026-11-20 17:20:56.20842|
CUST007 |Lakshmi Rao |lakshmi.rao7@mailexample.in |Pune |2023-08-04 |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv|2026-11-20 17:20:56.20842|
CUST008 |Karthik Khan |invalid.email.example.in |Pune |2025-08-09 |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv|2026-11-20 17:20:56.20842|
CUST009 |Anita Das |anita.das9@mailexample.in |Mumbai |2025-01-10 |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv|2026-11-20 17:20:56.20842|
CUST010 |Anita Sharma |anita.sharma10@mailexample.in|hyderabad|2023-10-22 |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv|2026-11-20 17:20:56.20842|
+-----------+-------------+-----------------------------+---------+-----------+-------------------------------------------------------------------------------------------------------+-------------------------+
only showing top 10 rows
+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
product_id|product_name |category |unit_price|active|_source_file |_ingestion_timestamp |
+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
PRD001 |Wireless Earbuds |Electronics|2499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:00.169403|
PRD002 |Smart Watch |Electronics|4999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:00.169403|
PRD003 |Bluetooth Speaker |Electronics|1899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:00.169403|
PRD004 |Power Bank 20000mAh|Electronics|-1599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:00.169403|
PRD005 |Cotton T-Shirt |Apparel |599.00 |Y |abfs

In [0]:
%scala
customersDF.write
  .format("delta")
  .mode("overwrite")
  .save(customersTarget)

productsDF.write
  .format("delta")
  .mode("overwrite")
  .save(productsTarget)

ordersDF.write
  .format("delta")
  .mode("overwrite")
  .save(ordersTarget)

println("Bronze Delta ingestion completed!")

Bronze Delta ingestion completed!

In [0]:
%scala
val bronzeCustomers =
  spark.read.format("delta").load(customersTarget)

val bronzeProducts =
  spark.read.format("delta").load(productsTarget)

val bronzeOrders =
  spark.read.format("delta").load(ordersTarget)

println("Customers: " + bronzeCustomers.count())
println("Products: " + bronzeProducts.count())
println("Orders: " + bronzeOrders.count())

Customers: 42
Products: 27
Orders: 161
val bronzeCustomers: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 5 more fields]
val bronzeProducts: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 5 more fields]
val bronzeOrders: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 7 more fields]

In [0]:
%scala
display(bronzeCustomers)

customer_id,customer_name,email,city,signup_date,_source_file,_ingestion_timestamp
CUST001,Priya Patel,priya.patel1@mailexample.in,Mumbai,2025-06-24,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST002,Sneha Gupta,sneha.gupta2@mailexample.in,Kolkata,2025-03-13,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST003,Divya Singh,divya.singh3@mailexample.in,Kochi,2025-02-08,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST004,Suresh Gupta,suresh.gupta4@mailexample.in,Kochi,2023-09-03,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST005,Aarav Gupta,aarav.gupta5@mailexample.in,Mumbai,2023-04-24,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST006,Vikram Sharma,vikram.sharma6@mailexample.in,Kochi,2023-01-12,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST007,Lakshmi Rao,lakshmi.rao7@mailexample.in,Pune,2023-08-04,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST008,Karthik Khan,invalid.email.example.in,Pune,2025-08-09,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST009,Anita Das,anita.das9@mailexample.in,Mumbai,2025-01-10,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST010,Anita Sharma,anita.sharma10@mailexample.in,hyderabad,2023-10-22,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z


In [0]:
%scala
display(bronzeProducts)

product_id,product_name,category,unit_price,active,_source_file,_ingestion_timestamp
PRD001,Wireless Earbuds,Electronics,2499.00,Y,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,2026-11-20T17:21:43.773662Z
PRD002,Smart Watch,Electronics,4999.00,Y,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,2026-11-20T17:21:43.773662Z
PRD003,Bluetooth Speaker,Electronics,1899.00,Y,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,2026-11-20T17:21:43.773662Z
PRD004,Power Bank 20000mAh,Electronics,-1599.00,Y,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,2026-11-20T17:21:43.773662Z
PRD005,Cotton T-Shirt,Apparel,599.00,Y,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,2026-11-20T17:21:43.773662Z
PRD006,Denim Jeans,Apparel,1799.00,Y,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,2026-11-20T17:21:43.773662Z
PRD007,Running Shoes,Apparel,3299.00,Y,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,2026-11-20T17:21:43.773662Z
PRD008,Hoodie,Apparel,1499.00,Y,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,2026-11-20T17:21:43.773662Z
PRD009,Air Fryer,Home,0,Y,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,2026-11-20T17:21:43.773662Z
PRD010,Cookware Set,Home,2999.00,Y,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv,2026-11-20T17:21:43.773662Z


In [0]:
%scala
display(bronzeOrders)

order_id,customer_id,product_id,quantity,order_date,order_status,last_updated,_source_file,_ingestion_timestamp
ORD000026,CUST020,PRD004,1,2026-09-08,CANCELLED,2026-09-08 09:15:01,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z
ORD000103,CUST018,PRD012,4,2026-09-02,RETURNED,2026-09-03 15:33:29,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z
ORD000089,CUST010,PRD005,2,2026-09-20,CANCELLED,2026-09-20 17:35:45,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z
ORD000032,CUST022,PRD007,5,2026-11-14,DELIVERED,2026-11-14 22:01:34,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z
ORD000138,CUST007,PRD008,4,2026-09-23,RETURNED,2026-09-23 19:29:37,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z
ORD000047,CUST038,PRD010,5,2026-11-08,DELIVERED,2026-11-08 11:54:54,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z
ORD000038,CUST017,PRD011,2,2026-10-06,PLACED,2026-10-06 16:30:26,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z
ORD000073,CUST013,PRD019,1,null,DELIVERED,2026-10-18 09:35:44,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z
ORD000095,CUST004,PRD004,3,2026-11-05,PLACED,2026-11-05 08:30:00,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z
ORD000065,CUST036,PRD004,3,2026-10-05,DELIVERED,2026-10-05 11:21:09,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z


Customer Silver transformation

Read Bronze customers

In [0]:
%scala
val bronzeCustomers =
  spark.read
    .format("delta")
    .load(basePath + "bronze_delta/customers")

display(bronzeCustomers)

customer_id,customer_name,email,city,signup_date,_source_file,_ingestion_timestamp
CUST001,Priya Patel,priya.patel1@mailexample.in,Mumbai,2025-06-24,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST002,Sneha Gupta,sneha.gupta2@mailexample.in,Kolkata,2025-03-13,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST003,Divya Singh,divya.singh3@mailexample.in,Kochi,2025-02-08,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST004,Suresh Gupta,suresh.gupta4@mailexample.in,Kochi,2023-09-03,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST005,Aarav Gupta,aarav.gupta5@mailexample.in,Mumbai,2023-04-24,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST006,Vikram Sharma,vikram.sharma6@mailexample.in,Kochi,2023-01-12,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST007,Lakshmi Rao,lakshmi.rao7@mailexample.in,Pune,2023-08-04,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST008,Karthik Khan,invalid.email.example.in,Pune,2025-08-09,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST009,Anita Das,anita.das9@mailexample.in,Mumbai,2025-01-10,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z
CUST010,Anita Sharma,anita.sharma10@mailexample.in,hyderabad,2023-10-22,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv,2026-11-20T17:21:39.076586Z


Clean and standardize columns

In [0]:
%scala
spark.sql("""
  SELECT
    try_to_timestamp('15/03/2024', 'dd/MM/yyyy') AS parsed_date
""").show(false)

+-------------------+
parsed_date |
+-------------------+
2024-03-15 00:00:00|
+-------------------+

In [0]:
%scala
import org.apache.spark.sql.functions._

val cleanedCustomersV2 = bronzeCustomers
  .withColumn("customer_id", upper(trim(col("customer_id"))))
  .withColumn("customer_name", trim(col("customer_name")))
  .withColumn("email", lower(trim(col("email"))))
  .withColumn("city", initcap(lower(trim(col("city")))))
  .withColumn(
    "signup_date_parsed",
    coalesce(
      to_date(try_to_timestamp(col("signup_date"), lit("yyyy-MM-dd"))),
      to_date(try_to_timestamp(col("signup_date"), lit("dd/MM/yyyy")))
    )
  )

import org.apache.spark.sql.functions._
val cleanedCustomersV2: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 6 more fields]

Apply validation rules

In [0]:
%scala
val validatedCustomersV2 = cleanedCustomersV2
  .withColumn(
    "rejection_reason",
    concat_ws(
      "; ",
      when(
        col("customer_id").isNull || col("customer_id") === "",
        lit("Missing customer ID")
      ),
      when(
        col("customer_name").isNull || col("customer_name") === "",
        lit("Missing customer name")
      ),
      when(
        col("city").isNull || col("city") === "",
        lit("Missing city")
      ),
      when(
        col("signup_date").isNull || trim(col("signup_date")) === "",
        lit("Missing signup date")
      ),
      when(
        col("signup_date").isNotNull &&
        trim(col("signup_date")) =!= "" &&
        col("signup_date_parsed").isNull,
        lit("Invalid signup date")
      ),
      when(
        col("signup_date_parsed") > current_date(),
        lit("Future signup date")
      ),
      when(
        col("email").isNotNull &&
        col("email") =!= "" &&
        !col("email").rlike("^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"),
        lit("Invalid email")
      )
    )
  )

println("Customer validation completed")

Customer validation completed
val validatedCustomersV2: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 7 more fields]

Separate valid and rejected records

In [0]:
%scala
val customerRejectedV2 = validatedCustomersV2
  .filter(col("rejection_reason") =!= "")

val customerValidV2 = validatedCustomersV2
  .filter(col("rejection_reason") === "")

val customerRejectedV2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: string, customer_name: string ... 7 more fields]
val customerValidV2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: string, customer_name: string ... 7 more fields]

In [0]:
%scala
cleanedCustomersV2
  .filter(col("signup_date") === "15/03/2024")
  .select("customer_id", "signup_date", "signup_date_parsed")
  .show(false)

+-----------+-----------+------------------+
customer_id|signup_date|signup_date_parsed|
+-----------+-----------+------------------+
CUST022 |15/03/2024 |2024-03-15 |
+-----------+-----------+------------------+

In [0]:
%scala
validatedCustomersV2
  .groupBy("rejection_reason")
  .count()
  .show(false)

+---------------------+-----+
rejection_reason |count|
+---------------------+-----+
Invalid email |2 |
Future signup date |1 |
 |38 |
Missing customer name|1 |
+---------------------+-----+

Inspect rejected records

In [0]:
%scala
customerRejectedV2.select(
  "customer_id",
  "customer_name",
  "email",
  "city",
  "signup_date",
  "rejection_reason"
).show(50, false)

+-----------+-------------+------------------------------+---------+-----------+---------------------+
customer_id|customer_name|email |city |signup_date|rejection_reason |
+-----------+-------------+------------------------------+---------+-----------+---------------------+
CUST008 |Karthik Khan |invalid.email.example.in |Pune |2025-08-09 |Invalid email |
CUST017 |NULL |karthik.gupta17@mailexample.in|Kochi |2024-06-09 |Missing customer name|
CUST026 |Vikram Khan |no@domain |Hyderabad|2023-02-03 |Invalid email |
CUST034 |Rahul Iyer |rahul.iyer34@mailexample.in |Kolkata |2027-06-01 |Future signup date |
+-----------+-------------+------------------------------+---------+-----------+---------------------+

In [0]:
%scala
println("Bronze customers: " + bronzeCustomers.count())
println("Valid customers: " + customerValidV2.count())
println("Rejected customers: " + customerRejectedV2.count())

Bronze customers: 42
Valid customers: 38
Rejected customers: 4

Customer deduplication

In [0]:
%scala
import org.apache.spark.sql.expressions.Window

val customerWindowV2 = Window
  .partitionBy("customer_id")
  .orderBy(
    col("signup_date_parsed").desc_nulls_last,
    col("customer_name").asc,
    col("email").asc,
    col("city").asc
  )

val rankedCustomersV2 = customerValidV2
  .withColumn("row_num", row_number().over(customerWindowV2))

val silverCustomersV2 = rankedCustomersV2
  .filter(col("row_num") === 1)
  .drop("row_num", "rejection_reason")

val duplicateCustomersV2 = rankedCustomersV2
  .filter(col("row_num") > 1)
  .withColumn("rejection_reason", lit("Duplicate customer ID"))
  .drop("row_num")

import org.apache.spark.sql.expressions.Window
val customerWindowV2: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@47d89703
val rankedCustomersV2: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 8 more fields]
val silverCustomersV2: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 6 more fields]
val duplicateCustomersV2: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 7 more fields]

In [0]:
%scala
println("Valid before deduplication: " + customerValidV2.count())
println("Silver after deduplication: " + silverCustomersV2.count())
println("Duplicate records: " + duplicateCustomersV2.count())

Valid before deduplication: 38
Silver after deduplication: 36
Duplicate records: 2

Combine all rejected customers

In [0]:
%scala
val allRejectedCustomersV2 = customerRejectedV2
  .unionByName(duplicateCustomersV2)

println("Total rejected customers: " + allRejectedCustomersV2.count())

allRejectedCustomersV2
  .select(
    "customer_id",
    "customer_name",
    "rejection_reason"
  )
  .show(false)

Total rejected customers: 6
+-----------+-------------+---------------------+
customer_id|customer_name|rejection_reason |
+-----------+-------------+---------------------+
CUST008 |Karthik Khan |Invalid email |
CUST017 |NULL |Missing customer name|
CUST026 |Vikram Khan |Invalid email |
CUST034 |Rahul Iyer |Future signup date |
CUST005 |Aarav Gupta |Duplicate customer ID|
CUST020 |Divya Das |Duplicate customer ID|
+-----------+-------------+---------------------+

val allRejectedCustomersV2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: string, customer_name: string ... 7 more fields]

Prepare the silver DataFrame

In [0]:
%scala
val silverCustomersFinal = silverCustomersV2.select(
  col("customer_id"),
  col("customer_name"),
  col("email"),
  col("city"),
  col("signup_date_parsed").as("signup_date"),
  col("_source_file"),
  col("_ingestion_timestamp")
)

silverCustomersFinal.printSchema()

println("Final Silver count: " + silverCustomersFinal.count())

root
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)
-- email: string (nullable = true)
-- city: string (nullable = true)
-- signup_date: date (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

Final Silver count: 36
val silverCustomersFinal: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 5 more fields]

Write Silver and Rejected Delta datasets to ADLS Gen2

In [0]:
%scala
val silverCustomerPath =
  basePath + "silver/customers"

silverCustomersFinal.write
  .format("delta")
  .mode("overwrite")
  .save(silverCustomerPath)

println("Silver Customers Delta write successful")

Silver Customers Delta write successful
val silverCustomerPath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/customers

In [0]:
%scala
val verifySilverCustomers = spark.read
  .format("delta")
  .load(silverCustomerPath)

println("Silver customer records: " + verifySilverCustomers.count())

verifySilverCustomers.printSchema()

Silver customer records: 36
root
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)
-- email: string (nullable = true)
-- city: string (nullable = true)
-- signup_date: date (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

val verifySilverCustomers: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 5 more fields]

Write rejected customers

In [0]:
%scala
val rejectedCustomerPath =
  basePath + "rejected/customers"

allRejectedCustomersV2.write
  .format("delta")
  .mode("overwrite")
  .save(rejectedCustomerPath)

println("Rejected Customers Delta write successful")

Rejected Customers Delta write successful
val rejectedCustomerPath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/rejected/customers

Verify rejected records

In [0]:
%scala
val verifyRejectedCustomers = spark.read
  .format("delta")
  .load(rejectedCustomerPath)

println("Rejected customer records: " + verifyRejectedCustomers.count())

verifyRejectedCustomers
  .select("customer_id", "customer_name", "rejection_reason")
  .show(false)

Rejected customer records: 6
+-----------+-------------+---------------------+
customer_id|customer_name|rejection_reason |
+-----------+-------------+---------------------+
CUST008 |Karthik Khan |Invalid email |
CUST017 |NULL |Missing customer name|
CUST026 |Vikram Khan |Invalid email |
CUST034 |Rahul Iyer |Future signup date |
CUST005 |Aarav Gupta |Duplicate customer ID|
CUST020 |Divya Das |Duplicate customer ID|
+-----------+-------------+---------------------+

val verifyRejectedCustomers: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 7 more fields]

Read Bronze Products Delta

In [0]:
%scala
val bronzeProductsV2 = spark.read
  .format("delta")
  .load(basePath + "bronze_delta/products")

println("Total Bronze Products: " + bronzeProductsV2.count())

bronzeProductsV2.printSchema()

bronzeProductsV2.show(false)

Total Bronze Products: 27
root
-- product_id: string (nullable = true)
-- product_name: string (nullable = true)
-- category: string (nullable = true)
-- unit_price: string (nullable = true)
-- active: string (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
product_id|product_name |category |unit_price|active|_source_file |_ingestion_timestamp |
+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
PRD001 |Wireless Earbuds |Electronics|2499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD002 |Smart Watch |Electronics|4999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD003 |Bluetooth Speaker |Electronics|1899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD004 |Power Bank 20000mAh|Electronics|-1599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD005 |Cotton T-Shirt |Apparel |599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD006 |Denim Jeans |Apparel |1799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD007 |Running Shoes |Apparel |3299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD008 |Hoodie |Apparel |1499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD009 |Air Fryer |Home |0 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD010 |Cookware Set |Home |2999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD011 |LED Desk Lamp |Home |899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD012 |Bedsheet Set |Home |1299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD013 |Basmati Rice 5kg |Grocery |649.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD014 |Cold Pressed Oil 1L|Grocery |449.00 |N |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD015 |Assorted Nuts 500g |Grocery |699.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD016 |Green Tea 100 bags |NULL |349.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD017 |Face Serum |Beauty |799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD018 |Sunscreen SPF50 |home |449.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD019 |Shampoo 650ml |Beauty |399.00 |Y |abfss://adlsdatabrics2c@adlsdatabric

Inspect all 27 records

In [0]:
%scala
bronzeProductsV2
  .orderBy("product_id")
  .show(100, false)

+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
product_id|product_name |category |unit_price|active|_source_file |_ingestion_timestamp |
+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
PRD001 |Wireless Earbuds |Electronics|2499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD002 |Smart Watch |Electronics|4999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD003 |Bluetooth Speaker |Electronics|1899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD004 |Power Bank 20000mAh|Electronics|-1599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD005 |Cotton T-Shirt |Apparel |599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD006 |Denim Jeans |Apparel |1799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD007 |Running Shoes |Apparel |3299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD007 |Running Shoes |Apparel |3299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD008 |Hoodie |Apparel |1499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD009 |Air Fryer |Home |0 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD010 |Cookware Set |Home |2999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD011 |LED Desk Lamp |Home |899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD011 |LED Desk Lamp |Home |949.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD012 |Bedsheet Set |Home |1299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD013 |Basmati Rice 5kg |Grocery |649.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD014 |Cold Pressed Oil 1L|Grocery |449.00 |N |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD015 |Assorted Nuts 500g |Grocery |699.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD016 |Green Tea 100 bags |NULL |349.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD017 |Face Serum |Beauty |799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD018 |Sunscreen SPF50 |home |449.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD019 |Shampoo 650ml |Beauty |399.00 |Y |abfss://adlsdat

Check duplicate product IDs

In [0]:
%scala
bronzeProductsV2
  .groupBy("product_id")
  .count()
  .filter(col("count") > 1)
  .show(false)

+----------+-----+
product_id|count|
+----------+-----+
PRD011 |2 |
PRD007 |2 |
+----------+-----+

inspect all 27 product records

In [0]:
%scala
bronzeProductsV2
  .orderBy("product_id")
  .show(100, false)

+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
product_id|product_name |category |unit_price|active|_source_file |_ingestion_timestamp |
+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
PRD001 |Wireless Earbuds |Electronics|2499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD002 |Smart Watch |Electronics|4999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD003 |Bluetooth Speaker |Electronics|1899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD004 |Power Bank 20000mAh|Electronics|-1599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD005 |Cotton T-Shirt |Apparel |599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD006 |Denim Jeans |Apparel |1799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD007 |Running Shoes |Apparel |3299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD007 |Running Shoes |Apparel |3299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD008 |Hoodie |Apparel |1499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD009 |Air Fryer |Home |0 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD010 |Cookware Set |Home |2999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD011 |LED Desk Lamp |Home |899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD011 |LED Desk Lamp |Home |949.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD012 |Bedsheet Set |Home |1299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD013 |Basmati Rice 5kg |Grocery |649.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD014 |Cold Pressed Oil 1L|Grocery |449.00 |N |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD015 |Assorted Nuts 500g |Grocery |699.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD016 |Green Tea 100 bags |NULL |349.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD017 |Face Serum |Beauty |799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD018 |Sunscreen SPF50 |home |449.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD019 |Shampoo 650ml |Beauty |399.00 |Y |abfss://adlsdat

Clean and standardize product data

In [0]:
%scala
import org.apache.spark.sql.functions._

val cleanedProductsV2 = bronzeProductsV2
  .withColumn("product_id", upper(trim(col("product_id"))))
  .withColumn("product_name", trim(col("product_name")))
  .withColumn("category", initcap(lower(trim(col("category")))))
  .withColumn(
    "unit_price_parsed",
    expr("try_cast(trim(unit_price) as decimal(10,2))")
  )
  .withColumn("active", upper(trim(col("active"))))

cleanedProductsV2.show(100, false)
cleanedProductsV2.printSchema()

+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+-----------------+
product_id|product_name |category |unit_price|active|_source_file |_ingestion_timestamp |unit_price_parsed|
+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+-----------------+
PRD001 |Wireless Earbuds |Electronics|2499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|2499.00 |
PRD002 |Smart Watch |Electronics|4999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|4999.00 |
PRD003 |Bluetooth Speaker |Electronics|1899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|1899.00 |
PRD004 |Power Bank 20000mAh|Electronics|-1599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|-1599.00 |
PRD005 |Cotton T-Shirt |Apparel |599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|599.00 |
PRD006 |Denim Jeans |Apparel |1799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|1799.00 |
PRD007 |Running Shoes |Apparel |3299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|3299.00 |
PRD008 |Hoodie |Apparel |1499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|1499.00 |
PRD009 |Air Fryer |Home |0 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|0.00 |
PRD010 |Cookware Set |Home |2999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|2999.00 |
PRD011 |LED Desk Lamp |Home |899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|899.00 |
PRD012 |Bedsheet Set |Home |1299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|1299.00 |
PRD013 |Basmati Rice 5kg |Grocery |649.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|649.00 |
PRD014 |Cold Pressed Oil 1L|Grocery |449.00 |N |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|449.00 |
PRD015 |Assorted Nuts 500g |Grocery |699.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|699.00 |
PRD016 |Green Tea 100 bags |NULL |349.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|349.00 |
PRD017 |Face Serum |Beauty |799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|799.00 |
PRD018 |Sunscreen SPF50 |Home |449.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|449.00 |
PRD019 |Shampoo 650ml |Beauty |399.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|399.00 |
PRD020 |Perfume

Validate product records

In [0]:
%scala
val validatedProductsV2 = cleanedProductsV2
  .withColumn(
    "rejection_reason",
    concat_ws(
      "; ",
      when(
        col("product_id").isNull || col("product_id") === "",
        lit("Missing product ID")
      ),
      when(
        col("product_name").isNull || col("product_name") === "",
        lit("Missing product name")
      ),
      when(
        col("category").isNull || col("category") === "",
        lit("Missing category")
      ),
      when(
        col("unit_price").isNull || trim(col("unit_price")) === "",
        lit("Missing unit price")
      ),
      when(
        col("unit_price").isNotNull &&
        trim(col("unit_price")) =!= "" &&
        col("unit_price_parsed").isNull,
        lit("Invalid unit price")
      ),
      when(
        col("unit_price_parsed") <= 0,
        lit("Unit price must be greater than zero")
      ),
      when(
        col("active").isNull || !col("active").isin("Y", "N"),
        lit("Invalid active flag")
      )
    )
  )

val productRejectedV2 = validatedProductsV2
  .filter(col("rejection_reason") =!= "")

val productValidV2 = validatedProductsV2
  .filter(col("rejection_reason") === "")

println("Total Bronze: " + bronzeProductsV2.count())
println("Validation Valid: " + productValidV2.count())
println("Validation Rejected: " + productRejectedV2.count())

productRejectedV2
  .select("product_id", "product_name", "unit_price", "rejection_reason")
  .show(false)

Total Bronze: 27
Validation Valid: 23
Validation Rejected: 4
+----------+-------------------+----------+------------------------------------+
product_id|product_name |unit_price|rejection_reason |
+----------+-------------------+----------+------------------------------------+
PRD004 |Power Bank 20000mAh|-1599.00 |Unit price must be greater than zero|
PRD009 |Air Fryer |0 |Unit price must be greater than zero|
PRD016 |Green Tea 100 bags |349.00 |Missing category |
PRD999 |NULL |100.00 |Missing product name |
+----------+-------------------+----------+------------------------------------+

val validatedProductsV2: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 7 more fields]
val productRejectedV2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product_id: string, product_name: string ... 7 more fields]
val productValidV2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product_id: string, product_name: string ... 7 more fields]

Detect duplicate and conflicting product IDs

In [0]:
%scala
import org.apache.spark.sql.expressions.Window

// Identify product IDs that have multiple records
val duplicateProductIds = productValidV2
  .groupBy("product_id")
  .count()
  .filter(col("count") > 1)
  .select("product_id")

// Identify IDs whose records have different business values
val conflictingProductIds = productValidV2
  .groupBy("product_id")
  .agg(
    countDistinct(
      struct(
        col("product_name"),
        col("category"),
        col("unit_price_parsed"),
        col("active")
      )
    ).as("distinct_versions")
  )
  .filter(col("distinct_versions") > 1)
  .select("product_id")

println("Duplicate product IDs:")
duplicateProductIds.show(false)

println("Conflicting product IDs:")
conflictingProductIds.show(false)

Duplicate product IDs:
+----------+
product_id|
+----------+
PRD011 |
PRD007 |
+----------+

Conflicting product IDs:
+----------+
product_id|
+----------+
PRD011 |
+----------+

import org.apache.spark.sql.expressions.Window
val duplicateProductIds: org.apache.spark.sql.DataFrame = [product_id: string]
val conflictingProductIds: org.apache.spark.sql.DataFrame = [product_id: string]

Separate exact duplicates and conflicting records

In [0]:
%scala

// Remove all conflicting product IDs from the valid dataset
val productsWithoutConflicts = productValidV2
  .join(conflictingProductIds, Seq("product_id"), "left_anti")

// Rank remaining records by product ID
val productWindow = Window
  .partitionBy("product_id")
  .orderBy(
    col("product_name").asc,
    col("category").asc,
    col("unit_price_parsed").asc,
    col("active").asc
  )

val rankedProducts = productsWithoutConflicts
  .withColumn("row_num", row_number().over(productWindow))

// Keep one record per product ID
val silverProductsV2 = rankedProducts
  .filter(col("row_num") === 1)
  .drop("row_num", "rejection_reason")

// Capture extra exact duplicate records
val duplicateProductsRejected = rankedProducts
  .filter(col("row_num") > 1)
  .withColumn("rejection_reason", lit("Duplicate product ID"))
  .drop("row_num")

// Capture both records of conflicting product IDs
val conflictingProductsRejected = productValidV2
  .join(conflictingProductIds, Seq("product_id"), "inner")
  .withColumn(
    "rejection_reason",
    lit("Conflicting duplicate product ID")
  )

println("Silver products: " + silverProductsV2.count())
println("Exact duplicate rejected: " + duplicateProductsRejected.count())
println("Conflicting records rejected: " + conflictingProductsRejected.count())

duplicateProductsRejected
  .select("product_id", "product_name", "unit_price", "rejection_reason")
  .show(false)

conflictingProductsRejected
  .select("product_id", "product_name", "unit_price", "rejection_reason")
  .show(false)

Silver products: 20
Exact duplicate rejected: 1
Conflicting records rejected: 2
+----------+-------------+----------+--------------------+
product_id|product_name |unit_price|rejection_reason |
+----------+-------------+----------+--------------------+
PRD007 |Running Shoes|3299.00 |Duplicate product ID|
+----------+-------------+----------+--------------------+

+----------+-------------+----------+--------------------------------+
product_id|product_name |unit_price|rejection_reason |
+----------+-------------+----------+--------------------------------+
PRD011 |LED Desk Lamp|899.00 |Conflicting duplicate product ID|
PRD011 |LED Desk Lamp|949.00 |Conflicting duplicate product ID|
+----------+-------------+----------+--------------------------------+

val productsWithoutConflicts: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 7 more fields]
val productWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@42e4012f
val rankedProducts: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 8 more fields]
val silverProductsV2: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 6 more fields]
val duplicateProductsRejected: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 7 more fields]
val conflictingProductsRejected: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 7 more fields]

Combine all rejected product records

In [0]:
%scala

val allRejectedProductsV2 = productRejectedV2
  .unionByName(duplicateProductsRejected)
  .unionByName(conflictingProductsRejected)

println("Total rejected products: " + allRejectedProductsV2.count())

allRejectedProductsV2
  .select(
    "product_id",
    "product_name",
    "category",
    "unit_price",
    "rejection_reason"
  )
  .orderBy("product_id")
  .show(100, false)

Total rejected products: 7
+----------+-------------------+-----------+----------+------------------------------------+
product_id|product_name |category |unit_price|rejection_reason |
+----------+-------------------+-----------+----------+------------------------------------+
PRD004 |Power Bank 20000mAh|Electronics|-1599.00 |Unit price must be greater than zero|
PRD007 |Running Shoes |Apparel |3299.00 |Duplicate product ID |
PRD009 |Air Fryer |Home |0 |Unit price must be greater than zero|
PRD011 |LED Desk Lamp |Home |899.00 |Conflicting duplicate product ID |
PRD011 |LED Desk Lamp |Home |949.00 |Conflicting duplicate product ID |
PRD016 |Green Tea 100 bags |NULL |349.00 |Missing category |
PRD999 |NULL |Misc |100.00 |Missing product name |
+----------+-------------------+-----------+----------+------------------------------------+

val allRejectedProductsV2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product_id: string, product_name: string ... 7 more fields]

Prepare final Silver product schema

In [0]:
%scala

val silverProductsFinal = silverProductsV2.select(
  col("product_id"),
  col("product_name"),
  col("category"),
  col("unit_price_parsed").as("unit_price"),
  col("active"),
  col("_source_file"),
  col("_ingestion_timestamp")
)

println("Final Silver Product Count: " + silverProductsFinal.count())

silverProductsFinal.printSchema()

silverProductsFinal
  .orderBy("product_id")
  .show(100, false)

Final Silver Product Count: 20
root
-- product_id: string (nullable = true)
-- product_name: string (nullable = true)
-- category: string (nullable = true)
-- unit_price: decimal(10,2) (nullable = true)
-- active: string (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
product_id|product_name |category |unit_price|active|_source_file |_ingestion_timestamp |
+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
PRD001 |Wireless Earbuds |Electronics|2499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD002 |Smart Watch |Electronics|4999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD003 |Bluetooth Speaker |Electronics|1899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD005 |Cotton T-Shirt |Apparel |599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD006 |Denim Jeans |Apparel |1799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD007 |Running Shoes |Apparel |3299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD008 |Hoodie |Apparel |1499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD010 |Cookware Set |Home |2999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD012 |Bedsheet Set |Home |1299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD013 |Basmati Rice 5kg |Grocery |649.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD014 |Cold Pressed Oil 1L|Grocery |449.00 |N |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD015 |Assorted Nuts 500g |Grocery |699.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD017 |Face Serum |Beauty |799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD018 |Sunscreen SPF50 |Home |449.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD019 |Shampoo 650ml |Beauty |399.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD020 |Perfume 100ml |Beauty |1999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD021 |Yoga Mat |Sports |699.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD022 |Dumbbell Set 10kg |Sports |1899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD023 |Cricket Bat |Sports |2799.00 |N |abfss://adlsdatabrics2c@a

Save Silver and Rejected products to ADLS Gen2

In [0]:
%scala

// Define ADLS Gen2 paths
val silverProductPath = basePath + "silver/products"
val rejectedProductPath = basePath + "rejected/products"

// Write Silver Products
silverProductsFinal.write
  .format("delta")
  .mode("overwrite")
  .save(silverProductPath)

println("Silver Products written successfully")

// Write Rejected Products
allRejectedProductsV2.write
  .format("delta")
  .mode("overwrite")
  .save(rejectedProductPath)

println("Rejected Products written successfully")

Silver Products written successfully
Rejected Products written successfully
val silverProductPath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/products
val rejectedProductPath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/rejected/products

Read back and verify persisted data

In [0]:
%scala

val silverProductsCheck = spark.read
  .format("delta")
  .load(silverProductPath)

val rejectedProductsCheck = spark.read
  .format("delta")
  .load(rejectedProductPath)

println("========== PRODUCTS PIPELINE SUMMARY ==========")
println("Bronze Products   : " + bronzeProductsV2.count())
println("Silver Products   : " + silverProductsCheck.count())
println("Rejected Products : " + rejectedProductsCheck.count())

println("========== SILVER VERIFICATION ==========")
silverProductsCheck.printSchema()
silverProductsCheck.orderBy("product_id").show(100, false)

println("========== REJECTED VERIFICATION ==========")
rejectedProductsCheck
  .select("product_id", "rejection_reason")
  .orderBy("product_id")
  .show(100, false)

========== PRODUCTS PIPELINE SUMMARY ==========
Bronze Products : 27
Silver Products : 20
Rejected Products : 7
========== SILVER VERIFICATION ==========
root
-- product_id: string (nullable = true)
-- product_name: string (nullable = true)
-- category: string (nullable = true)
-- unit_price: decimal(10,2) (nullable = true)
-- active: string (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
product_id|product_name |category |unit_price|active|_source_file |_ingestion_timestamp |
+----------+-------------------+-----------+----------+------+-----------------------------------------------------------------------------------------------------+--------------------------+
PRD001 |Wireless Earbuds |Electronics|2499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD002 |Smart Watch |Electronics|4999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD003 |Bluetooth Speaker |Electronics|1899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD005 |Cotton T-Shirt |Apparel |599.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD006 |Denim Jeans |Apparel |1799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD007 |Running Shoes |Apparel |3299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD008 |Hoodie |Apparel |1499.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD010 |Cookware Set |Home |2999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD012 |Bedsheet Set |Home |1299.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD013 |Basmati Rice 5kg |Grocery |649.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD014 |Cold Pressed Oil 1L|Grocery |449.00 |N |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD015 |Assorted Nuts 500g |Grocery |699.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD017 |Face Serum |Beauty |799.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD018 |Sunscreen SPF50 |Home |449.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD019 |Shampoo 650ml |Beauty |399.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD020 |Perfume 100ml |Beauty |1999.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD021 |Yoga Mat |Sports |699.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv|2026-11-20 17:21:43.773662|
PRD022 |Dumbbell Set 10kg |Sports |1899.00 |Y |abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/

Orders Bronze-to-Silver pipeline

In [0]:
%scala

val bronzeOrdersV2 = spark.read
  .format("delta")
  .load(basePath + "bronze_delta/orders")

println("Total Bronze Orders: " + bronzeOrdersV2.count())

bronzeOrdersV2.printSchema()

bronzeOrdersV2
  .orderBy("order_id")
  .show(200, false)

Total Bronze Orders: 161
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: string (nullable = true)
-- order_date: string (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

+---------+-----------+----------+--------+----------+------------+-------------------+-------------------------------------------------------------------------------------------------+--------------------------+
order_id |customer_id|product_id|quantity|order_date|order_status|last_updated |_source_file |_ingestion_timestamp |
+---------+-----------+----------+--------+----------+------------+-------------------+-------------------------------------------------------------------------------------------------+--------------------------+
ORD000001|CUST023 |PRD005 |1 |2026-09-06|DELIVERED |2026-09-06 20:34:17|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000002|CUST032 |PRD005 |5 |2026-11-14|DELIVERED |2026-11-14 17:49:59|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000003|CUST033 |PRD010 |1 |2026-10-31|DELIVERED |2026-10-31 17:08:38|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000004|CUST038 |5 |5 |2026-09-21|DELIVERED |2026-09-21 22:08:50|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000005|CUST013 |PRD013 |1 |2026-10-20|DELIVERED |2026-10-20 20:09:33|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000006|CUST023 |PRD022 |0 |2026-10-20|RETURNED |2026-10-20 22:04:03|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000007|CUST021 |PRD024 |3 |2026-09-04|DELIVERED |2026-09-04 08:32:30|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000008|CUST030 |PRD020 |5 |2026-10-12|DELIVERED |2026-10-12 22:07:31|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000009|NULL |PRD021 |3 |2026-09-20|DELIVERED |2026-09-20 17:33:13|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000010|CUST005 |PRD019 |4 |2026-11-16|DELIVERED |2026-11-16 11:43:45|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000011|CUST003 |PRD022 |5 |2026-09-26|DELIVERED |2026-09-26 18:58:11|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000012|CUST013 |PRD017 |1 |2026-10-28|DELIVERED |2026-10-28 21:48:41|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000013|CUST034 |PRD777 |2 |2026-11-07|CANCELLED |2026-11-07 11:50:56|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000014|CUST035 |PRD015 |1 |2026-09-24|DELIVERED |2026-09-24 14:17:27|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000015|CUST017 |PRD016 |4 |2026-10-10|RETURNED |2026-10-10 10:22:11|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000016|CUST021 |PRD018 |4 |2026-10-24|DELIVERED |2026-10-24 12:29:11|abfss://adlsdatabr

Clean and parse Orders data

In [0]:
%scala
import org.apache.spark.sql.functions._

val cleanedOrdersV1 = bronzeOrdersV2
  .withColumn("order_id", upper(trim(col("order_id"))))
  .withColumn("customer_id", upper(trim(col("customer_id"))))
  .withColumn("product_id", upper(trim(col("product_id"))))
  .withColumn("order_status", upper(trim(col("order_status"))))
  .withColumn(
    "quantity_parsed",
    expr("try_cast(trim(quantity) AS INT)")
  )
  .withColumn(
    "order_date_parsed",
    coalesce(
      to_date(try_to_timestamp(col("order_date"), lit("yyyy-MM-dd"))),
      to_date(try_to_timestamp(col("order_date"), lit("dd-MM-yyyy")))
    )
  )
  .withColumn(
    "last_updated_parsed",
    try_to_timestamp(col("last_updated"), lit("yyyy-MM-dd HH:mm:ss"))
  )

cleanedOrdersV1.select(
  "order_id",
  "customer_id",
  "product_id",
  "quantity",
  "quantity_parsed",
  "order_date",
  "order_date_parsed",
  "order_status",
  "last_updated_parsed"
).show(30, false)

+---------+-----------+----------+--------+---------------+----------+-----------------+------------+-------------------+
order_id |customer_id|product_id|quantity|quantity_parsed|order_date|order_date_parsed|order_status|last_updated_parsed|
+---------+-----------+----------+--------+---------------+----------+-----------------+------------+-------------------+
ORD000026|CUST020 |PRD004 |1 |1 |2026-09-08|2026-09-08 |CANCELLED |2026-09-08 09:15:01|
ORD000103|CUST018 |PRD012 |4 |4 |2026-09-02|2026-09-02 |RETURNED |2026-09-03 15:33:29|
ORD000089|CUST010 |PRD005 |2 |2 |2026-09-20|2026-09-20 |CANCELLED |2026-09-20 17:35:45|
ORD000032|CUST022 |PRD007 |5 |5 |2026-11-14|2026-11-14 |DELIVERED |2026-11-14 22:01:34|
ORD000138|CUST007 |PRD008 |4 |4 |2026-09-23|2026-09-23 |RETURNED |2026-09-23 19:29:37|
ORD000047|CUST038 |PRD010 |5 |5 |2026-11-08|2026-11-08 |DELIVERED |2026-11-08 11:54:54|
ORD000038|CUST017 |PRD011 |2 |2 |2026-10-06|2026-10-06 |PLACED |2026-10-06 16:30:26|
ORD000073|CUST013 |PRD019 |1 |1 |NULL |NULL |DELIVERED |2026-10-18 09:35:44|
ORD000095|CUST004 |PRD004 |3 |3 |2026-11-05|2026-11-05 |PLACED |2026-11-05 08:30:00|
ORD000065|CUST036 |PRD004 |3 |3 |2026-10-05|2026-10-05 |DELIVERED |2026-10-05 11:21:09|
ORD000078|CUST021 |PRD009 |1 |1 |2026-11-17|2026-11-17 |CANCELLED |2026-11-17 15:16:31|
ORD000043|CUST036 |PRD017 |5 |5 |2026-10-16|2026-10-16 |DELIVERED |2026-10-16 15:40:29|
ORD000036|CUST888 |PRD001 |5 |5 |2026-09-29|2026-09-29 |CANCELLED |2026-09-29 20:08:11|
ORD000091|CUST023 |PRD005 |4 |4 |2026-09-18|2026-09-18 |DELIVERED |2026-09-18 12:34:44|
ORD000117|CUST038 |PRD001 |3 |3 |2026-10-11|2026-10-11 |PLACED |2026-10-11 12:49:27|
ORD000018|CUST022 |PRD015 |-2 |-2 |2026-09-01|2026-09-01 |CANCELLED |2026-09-01 09:56:29|
ORD000002|CUST032 |PRD005 |5 |5 |2026-11-14|2026-11-14 |DELIVERED |2026-11-14 17:49:59|
ORD000129|CUST012 |PRD006 |2 |2 |2026-09-12|2026-09-12 |DELIVERED |2026-09-12 15:43:39|
ORD000112|CUST030 |PRD008 |4 |4 |2026-10-31|2026-10-31 |DELIVERED |2026-10-31 11:20:45|
ORD000022|CUST017 |PRD004 |4 |4 |2026-11-11|2026-11-11 |DELIVERED |2026-11-11 10:06:50|
ORD000052|CUST035 |PRD006 |5 |5 |2026-11-01|2026-11-01 |DELIVERED |2026-11-01 14:22:51|
ORD000033|CUST008 |PRD002 |2 |2 |2026-09-15|2026-09-15 |SHIPPED |2026-09-15 10:39:07|
ORD000029|CUST008 |PRD024 |3 |3 |2026-10-30|2026-10-30 |SHIPPED |2026-10-30 14:43:56|
ORD000037|CUST010 |PRD008 |1 |1 |2026-11-11|2026-11-11 |DELIVERED |2026-11-11 14:56:16|
ORD000111|CUST008 |PRD011 |2 |2 |2026-09-10|2026-09-10 |DELIVERED |2026-09-10 20:38:33|
ORD000068|CUST023 |PRD022 |3 |3 |2026-11-09|2026-11-09 |DELIVERED |2026-11-09 16:13:27|
ORD000019|CUST011 |PRD012 |5 |5 |2026-09-24|2026-09-24 |CANCELLED |2026-09-24 08:22:54|
ORD000030|CUST023 |PRD002 |NULL |NULL |2026-10-03|2026-10-03 |DELIVERED |2026-10-03 14:18:52|
ORD000101|CUST028 |PRD007 |4 |4 |2026-09-21|2026-09-21 |CANCELLED |2026-09-21 17:11:13|
ORD000105|CUST036 |PRD011 |3 |3 |2026-11-17|2026-11-17 |CANCELLED |2026-11-17 18:37:03|
+---------+-----------+----------+--------+---------------+----------+-----------------+------------+-------------------+
only showing top 30 rows
import org.apache.spark.sql.functions._
val cleanedOrdersV1: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]

Validate Orders

In [0]:
%scala
val silverCustomersRef = spark.read
  .format("delta")
  .load(basePath + "silver/customers")
  .select("customer_id")
  .distinct()

val silverProductsRef = spark.read
  .format("delta")
  .load(basePath + "silver/products")
  .select("product_id")
  .distinct()

println("Valid Customer IDs: " + silverCustomersRef.count())
println("Valid Product IDs : " + silverProductsRef.count())

println("Customer reference sample:")
silverCustomersRef.show(10, false)

println("Product reference sample:")
silverProductsRef.show(10, false)

Valid Customer IDs: 36
Valid Product IDs : 20
Customer reference sample:
+-----------+
customer_id|
+-----------+
CUST004 |
CUST022 |
CUST007 |
CUST030 |
CUST010 |
CUST001 |
CUST002 |
CUST021 |
CUST006 |
CUST035 |
+-----------+
only showing top 10 rows
Product reference sample:
+----------+
product_id|
+----------+
PRD008 |
PRD014 |
PRD021 |
PRD005 |
PRD015 |
PRD002 |
PRD020 |
PRD018 |
PRD006 |
PRD023 |
+----------+
only showing top 10 rows
val silverCustomersRef: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: string]
val silverProductsRef: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product_id: string]

Apply validation rules

In [0]:
%scala
val ordersWithReferences = cleanedOrdersV1
  .join(
    silverCustomersRef
      .withColumnRenamed("customer_id", "_valid_customer_id"),
    cleanedOrdersV1("customer_id") === col("_valid_customer_id"),
    "left"
  )
  .join(
    silverProductsRef
      .withColumnRenamed("product_id", "_valid_product_id"),
    cleanedOrdersV1("product_id") === col("_valid_product_id"),
    "left"
  )

val validatedOrdersV1 = ordersWithReferences
  .withColumn(
    "rejection_reason",
    concat_ws(
      "; ",

      when(
        col("order_id").isNull || col("order_id") === "",
        lit("Missing order ID")
      ),

      when(
        col("order_id").isNotNull &&
        col("order_id") =!= "" &&
        !col("order_id").rlike("^ORD[0-9]{6}$"),
        lit("Invalid order ID format")
      ),

      when(
        col("customer_id").isNull || col("customer_id") === "",
        lit("Missing customer ID")
      ),

      when(
        col("customer_id").isNotNull &&
        col("customer_id") =!= "" &&
        col("_valid_customer_id").isNull,
        lit("Customer ID not found in Silver")
      ),

      when(
        col("product_id").isNull || col("product_id") === "",
        lit("Missing product ID")
      ),

      when(
        col("product_id").isNotNull &&
        col("product_id") =!= "" &&
        col("_valid_product_id").isNull,
        lit("Product ID not found in Silver")
      ),

      when(
        col("quantity").isNull || trim(col("quantity")) === "",
        lit("Missing quantity")
      ),

      when(
        col("quantity").isNotNull &&
        trim(col("quantity")) =!= "" &&
        col("quantity_parsed").isNull,
        lit("Invalid quantity")
      ),

      when(
        col("quantity_parsed") <= 0,
        lit("Quantity must be greater than zero")
      ),

      when(
        col("order_date").isNull || trim(col("order_date")) === "",
        lit("Missing order date")
      ),

      when(
        col("order_date").isNotNull &&
        trim(col("order_date")) =!= "" &&
        col("order_date_parsed").isNull,
        lit("Invalid order date")
      ),

      when(
        col("order_date_parsed") > current_date(),
        lit("Future order date")
      ),

      when(
        col("last_updated").isNull || trim(col("last_updated")) === "",
        lit("Missing last updated timestamp")
      ),

      when(
        col("last_updated").isNotNull &&
        trim(col("last_updated")) =!= "" &&
        col("last_updated_parsed").isNull,
        lit("Invalid last updated timestamp")
      ),

      when(
        col("order_status").isNull || col("order_status") === "",
        lit("Missing order status")
      ),

      when(
        col("order_status").isNotNull &&
        col("order_status") =!= "" &&
        !col("order_status").isin(
          "PLACED",
          "SHIPPED",
          "DELIVERED",
          "CANCELLED",
          "RETURNED"
        ),
        lit("Unrecognized order status")
      )
    )
  )

val orderValidationRejected = validatedOrdersV1
  .filter(col("rejection_reason") =!= "")

val orderValidationPassed = validatedOrdersV1
  .filter(col("rejection_reason") === "")

val ordersWithReferences: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 12 more fields]
val validatedOrdersV1: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 13 more fields]
val orderValidationRejected: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 13 more fields]
val orderValidationPassed: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 13 more fields]

Check validation results

In [0]:
%scala
println("========== ORDER VALIDATION SUMMARY ==========")

println("Total Bronze Orders     : " + cleanedOrdersV1.count())
println("Validation Passed       : " + orderValidationPassed.count())
println("Validation Rejected     : " + orderValidationRejected.count())

println("========== REJECTION REASONS ==========")

orderValidationRejected
  .select("order_id", "customer_id", "product_id", "quantity",
          "order_date", "order_status", "rejection_reason")
  .orderBy("order_id")
  .show(100, false)

========== ORDER VALIDATION SUMMARY ==========
Total Bronze Orders : 161
Validation Passed : 99
Validation Rejected : 62
========== REJECTION REASONS ==========
+---------+-----------+----------+--------+----------+------------+---------------------------------------------------------------+
order_id |customer_id|product_id|quantity|order_date|order_status|rejection_reason |
+---------+-----------+----------+--------+----------+------------+---------------------------------------------------------------+
ORD000004|CUST038 |5 |5 |2026-09-21|DELIVERED |Product ID not found in Silver |
ORD000006|CUST023 |PRD022 |0 |2026-10-20|RETURNED |Quantity must be greater than zero |
ORD000009|NULL |PRD021 |3 |2026-09-20|DELIVERED |Missing customer ID |
ORD000013|CUST034 |PRD777 |2 |2026-11-07|CANCELLED |Customer ID not found in Silver; Product ID not found in Silver|
ORD000015|CUST017 |PRD016 |4 |2026-10-10|RETURNED |Customer ID not found in Silver; Product ID not found in Silver|
ORD000017|CUST018 |PRD004 |2 |2026-09-16|DELIVERED |Product ID not found in Silver |
ORD000018|CUST022 |PRD015 |-2 |2026-09-01|CANCELLED |Quantity must be greater than zero |
ORD000020|CUST008 |PRD021 |3 |2026-11-13|SHIPPED |Customer ID not found in Silver |
ORD000022|CUST017 |PRD004 |4 |2026-11-11|DELIVERED |Customer ID not found in Silver; Product ID not found in Silver|
ORD000023|CUST999 |PRD006 |1 |2026-09-14|SHIPPED |Customer ID not found in Silver |
ORD000025|CUST027 |PRD009 |4 |2026-10-04|DELIVERED |Product ID not found in Silver |
ORD000026|CUST020 |PRD004 |1 |2026-09-08|CANCELLED |Product ID not found in Silver |
ORD000027|CUST023 |PRD011 |4 |2026-11-11|PLACED |Product ID not found in Silver |
ORD000029|CUST008 |PRD024 |3 |2026-10-30|SHIPPED |Customer ID not found in Silver |
ORD000030|CUST023 |PRD002 |NULL |2026-10-03|DELIVERED |Missing quantity |
ORD000031|CUST008 |PRD010 |4 |2026-10-07|DELIVERED |Customer ID not found in Silver |
ORD000033|CUST008 |PRD002 |2 |2026-09-15|SHIPPED |Customer ID not found in Silver |
ORD000036|CUST888 |PRD001 |5 |2026-09-29|CANCELLED |Customer ID not found in Silver |
ORD000038|CUST017 |PRD011 |2 |2026-10-06|PLACED |Customer ID not found in Silver; Product ID not found in Silver|
ORD000042|CUST018 |PRD011 |two |2026-10-17|DELIVERED |Product ID not found in Silver; Invalid quantity |
ORD000045|CUST003 |NULL |2 |2026-11-06|DELIVERED |Missing product ID |
ORD000046|CUST017 |PRD005 |1 |2026-10-09|DELIVERED |Customer ID not found in Silver |
ORD000049|CUST036 |PRD009 |3 |2026-09-19|DELIVERED |Product ID not found in Silver |
ORD000054|CUST024 |PRD010 |2 |2026-09-01|DLVRD |Unrecognized order status |
ORD000055|CUST009 |PRD004 |3 |2026-09-20|CANCELLED |Product ID not found in Silver |
ORD000057|CUST031 |PRD017 |1 |2026-10-09|UNKNOWN |Unrecognized order status |
ORD000061|CUST013 |PRD016 |2 |03-11-2026|CANCELLED |Product ID not found in Silver |
ORD000064|CUST029 |PRD009 |2 |2026-09-16|SHIPPED |Product ID not found in Silver |
ORD000065|CUST036 |PRD004 |3 |2026-10-05|DELIVERED |Product ID not found in Silver |
ORD000066|CUST026 |PRD001 |5 |2026-11-13|CANCELLED |Customer ID not found in Silver |
ORD000071|CUST008 |PRD024 |3 |2027-02-30|DELIVERED |Customer ID not found in Silver; Invalid order date |
ORD000072|CUST030 |PRD018 |3 |2026-12-25|PLACED |Future order date |
ORD000073|CUST013 |PRD019 |1 |NULL |DELIVERED |Missing order date |
ORD000075|CUST034 |PRD024 |5 |2026-10-10|SHIPPED |Customer ID not found in Silver |
ORD000077|CUST022 |PRD011 |3 |2026-11-05|DELIVERED |Product ID not found in Silver |
ORD000078|CUST021 |PRD009 |1 |2026-11-17|CANCELLED |Product ID not found in Silver |
ORD000079|CUST024 |PRD009 |4 |2026-11-02|DELIVERED |Product ID not found in Silver |
ORD000080|CUST026 |PRD010 |2 |2026-10-18|PLACED |Customer ID not found in Silver |
ORD000081|CUST005 |PRD022 |5 |2026-09-07|NULL |Missing order status |
ORD000084|CUST012 |PRD009 |5 |2026-09-24|SHIPPED |Product ID not found in Silver |
ORD000086|CUST034 |PRD005 |5 |2026-

In [0]:
%scala
duplicateOrderIds.show(50, false)

+---------+-----+
order_id |count|
+---------+-----+
ORD000095|2 |
ORD000104|2 |
ORD000106|2 |
ORD000105|2 |
ORD000102|2 |
ORD000092|2 |
ORD000093|2 |
ORD000091|2 |
ORD000101|2 |
ORD000094|2 |
ORD000103|2 |
+---------+-----+

Inspect duplicate order IDs

In [0]:
%scala
cleanedOrdersV1
  .filter(col("order_id") === "ORD000101")
  .select(
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "order_status",
    "last_updated",
    "last_updated_parsed"
  )
  .show(20, false)

+---------+-----------+----------+--------+------------+-------------------+-------------------+
order_id |customer_id|product_id|quantity|order_status|last_updated |last_updated_parsed|
+---------+-----------+----------+--------+------------+-------------------+-------------------+
ORD000101|CUST028 |PRD007 |4 |CANCELLED |2026-09-21 17:11:13|2026-09-21 17:11:13|
ORD000101|CUST028 |PRD007 |4 |DELIVERED |2026-09-21 09:11:13|2026-09-21 09:11:13|
+---------+-----------+----------+--------+------------+-------------------+-------------------+

Inspect all duplicate versions

In [0]:
%scala
cleanedOrdersV1
  .filter(col("order_id") === "ORD000091")
  .select("order_id", "order_status", "last_updated")
  .show(false)

+---------+------------+-------------------+
order_id |order_status|last_updated |
+---------+------------+-------------------+
ORD000091|DELIVERED |2026-09-18 12:34:44|
ORD000091|DELIVERED |2026-09-18 12:34:44|
+---------+------------+-------------------+

In [0]:
%scala
cleanedOrdersV1
  .filter(
    col("order_id") === "ORD000092" ||
    col("order_id") === "ORD000093" ||
    col("order_id") === "ORD000094" ||
    col("order_id") === "ORD000095"
  )
  .select("order_id", "quantity", "order_status", "last_updated")
  .show(20, false)

+---------+--------+------------+-------------------+
order_id |quantity|order_status|last_updated |
+---------+--------+------------+-------------------+
ORD000095|3 |PLACED |2026-11-05 08:30:00|
ORD000094|3 |DELIVERED |2026-11-09 09:29:43|
ORD000092|2 |DELIVERED |2026-09-24 19:03:48|
ORD000093|5 |DELIVERED |2026-11-02 22:34:01|
ORD000095|3 |PLACED |2026-11-05 08:30:00|
ORD000092|2 |DELIVERED |2026-09-24 19:03:48|
ORD000094|3 |DELIVERED |2026-11-09 09:29:43|
ORD000093|5 |DELIVERED |2026-11-02 22:34:01|
+---------+--------+------------+-------------------+

In [0]:
%scala
cleanedOrdersV1
  .filter(
    col("order_id") === "ORD000101" ||
    col("order_id") === "ORD000102" ||
    col("order_id") === "ORD000103" ||
    col("order_id") === "ORD000104" ||
    col("order_id") === "ORD000105" ||
    col("order_id") === "ORD000106"
  )
  .select("order_id", "quantity", "order_status", "last_updated")
  .show(20, false)

+---------+--------+------------+-------------------+
order_id |quantity|order_status|last_updated |
+---------+--------+------------+-------------------+
ORD000103|4 |RETURNED |2026-09-03 15:33:29|
ORD000101|4 |CANCELLED |2026-09-21 17:11:13|
ORD000105|3 |CANCELLED |2026-11-17 18:37:03|
ORD000104|4 |SHIPPED |2026-09-17 00:09:35|
ORD000104|4 |DELIVERED |2026-09-14 20:09:35|
ORD000106|5 |CANCELLED |2026-09-13 23:29:22|
ORD000102|3 |DELIVERED |2026-11-10 10:41:57|
ORD000103|4 |DELIVERED |2026-09-02 15:33:29|
ORD000105|4 |DELIVERED |2026-11-18 01:37:03|
ORD000102|3 |DELIVERED |2026-11-12 06:41:57|
ORD000101|4 |DELIVERED |2026-09-21 09:11:13|
ORD000106|5 |DELIVERED |2026-09-11 15:29:22|
+---------+--------+------------+-------------------+

Deduplicate using the latest update

In [0]:
%scala
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

// Remove exact duplicate business records
val distinctOrdersV1 = cleanedOrdersV1.dropDuplicates(
  Seq(
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "order_date",
    "order_status",
    "last_updated"
  )
)

// Orders without a valid update timestamp cannot be ranked
val ordersWithValidTimestamp = distinctOrdersV1
  .filter(col("last_updated_parsed").isNotNull)

// Find the latest timestamp for each order
val orderLatestWindow = Window.partitionBy("order_id")

val latestOrderCandidates = ordersWithValidTimestamp
  .withColumn(
    "max_last_updated",
    max(col("last_updated_parsed")).over(orderLatestWindow)
  )
  .filter(col("last_updated_parsed") === col("max_last_updated"))
  .drop("max_last_updated")

// Detect conflicting records with the same latest timestamp
val latestTimestampConflicts = latestOrderCandidates
  .groupBy("order_id")
  .count()
  .filter(col("count") > 1)

println("========== LATEST VERSION CHECK ==========")

println("Exact duplicate records removed: " +
  (cleanedOrdersV1.count() - distinctOrdersV1.count()))

println("Orders with latest timestamp: " +
  latestOrderCandidates.select("order_id").distinct().count())

println("Conflicting latest versions: " +
  latestTimestampConflicts.count())

latestTimestampConflicts.show(50, false)

========== LATEST VERSION CHECK ==========
Exact duplicate records removed: 5
Orders with latest timestamp: 150
Conflicting latest versions: 0
+--------+-----+
order_id|count|
+--------+-----+
+--------+-----+

import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._
val distinctOrdersV1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 10 more fields]
val ordersWithValidTimestamp: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 10 more fields]
val orderLatestWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@71614a9d
val latestOrderCandidates: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]
val latestTimestampConflicts: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, count: bigint]

Validate the latest order versions

In [0]:
%scala
val latestOrdersWithReferences = latestOrderCandidates
  .join(
    silverCustomersRef.withColumnRenamed(
      "customer_id", "_valid_customer_id"
    ),
    latestOrderCandidates("customer_id") === col("_valid_customer_id"),
    "left"
  )
  .join(
    silverProductsRef.withColumnRenamed(
      "product_id", "_valid_product_id"
    ),
    latestOrderCandidates("product_id") === col("_valid_product_id"),
    "left"
  )

val latestOrdersValidated = latestOrdersWithReferences
  .withColumn(
    "rejection_reason",
    concat_ws(
      "; ",
      when(col("order_id").isNull || col("order_id") === "",
        lit("Missing order ID")),
      when(col("customer_id").isNull || col("customer_id") === "",
        lit("Missing customer ID")),
      when(col("_valid_customer_id").isNull,
        lit("Customer ID not found in Silver")),
      when(col("product_id").isNull || col("product_id") === "",
        lit("Missing product ID")),
      when(col("_valid_product_id").isNull,
        lit("Product ID not found in Silver")),
      when(col("quantity_parsed").isNull,
        lit("Invalid or missing quantity")),
      when(col("quantity_parsed") <= 0,
        lit("Quantity must be greater than zero")),
      when(col("order_date_parsed").isNull,
        lit("Missing or invalid order date")),
      when(col("order_date_parsed") > current_date(),
        lit("Future order date")),
      when(col("last_updated_parsed").isNull,
        lit("Missing or invalid update timestamp")),
      when(col("order_status").isNull || col("order_status") === "",
        lit("Missing order status")),
      when(
        !col("order_status").isin(
          "PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"
        ),
        lit("Unrecognized order status")
      )
    )
  )

val silverOrdersCandidates = latestOrdersValidated
  .filter(col("rejection_reason") === "")

val rejectedLatestOrders = latestOrdersValidated
  .filter(col("rejection_reason") =!= "")

println("========== LATEST ORDER VALIDATION ==========")
println("Latest Candidates : " + latestOrderCandidates.count())
println("Valid Orders      : " + silverOrdersCandidates.count())
println("Rejected Orders   : " + rejectedLatestOrders.count())

rejectedLatestOrders
  .select("order_id", "customer_id", "product_id",
          "order_status", "last_updated", "rejection_reason")
  .orderBy("order_id")
  .show(100, false)

========== LATEST ORDER VALIDATION ==========
Latest Candidates : 150
Valid Orders : 90
Rejected Orders : 60
+---------+-----------+----------+------------+-------------------+---------------------------------------------------------------+
order_id |customer_id|product_id|order_status|last_updated |rejection_reason |
+---------+-----------+----------+------------+-------------------+---------------------------------------------------------------+
ORD000004|CUST038 |5 |DELIVERED |2026-09-21 22:08:50|Product ID not found in Silver |
ORD000006|CUST023 |PRD022 |RETURNED |2026-10-20 22:04:03|Quantity must be greater than zero |
ORD000009|NULL |PRD021 |DELIVERED |2026-09-20 17:33:13|Missing customer ID; Customer ID not found in Silver |
ORD000013|CUST034 |PRD777 |CANCELLED |2026-11-07 11:50:56|Customer ID not found in Silver; Product ID not found in Silver|
ORD000015|CUST017 |PRD016 |RETURNED |2026-10-10 10:22:11|Customer ID not found in Silver; Product ID not found in Silver|
ORD000017|CUST018 |PRD004 |DELIVERED |2026-09-16 16:50:33|Product ID not found in Silver |
ORD000018|CUST022 |PRD015 |CANCELLED |2026-09-01 09:56:29|Quantity must be greater than zero |
ORD000020|CUST008 |PRD021 |SHIPPED |2026-11-13 20:34:13|Customer ID not found in Silver |
ORD000022|CUST017 |PRD004 |DELIVERED |2026-11-11 10:06:50|Customer ID not found in Silver; Product ID not found in Silver|
ORD000023|CUST999 |PRD006 |SHIPPED |2026-09-14 18:03:48|Customer ID not found in Silver |
ORD000025|CUST027 |PRD009 |DELIVERED |2026-10-04 12:04:00|Product ID not found in Silver |
ORD000026|CUST020 |PRD004 |CANCELLED |2026-09-08 09:15:01|Product ID not found in Silver |
ORD000027|CUST023 |PRD011 |PLACED |2026-11-11 17:59:54|Product ID not found in Silver |
ORD000029|CUST008 |PRD024 |SHIPPED |2026-10-30 14:43:56|Customer ID not found in Silver |
ORD000030|CUST023 |PRD002 |DELIVERED |2026-10-03 14:18:52|Invalid or missing quantity |
ORD000031|CUST008 |PRD010 |DELIVERED |2026-10-07 19:21:18|Customer ID not found in Silver |
ORD000033|CUST008 |PRD002 |SHIPPED |2026-09-15 10:39:07|Customer ID not found in Silver |
ORD000036|CUST888 |PRD001 |CANCELLED |2026-09-29 20:08:11|Customer ID not found in Silver |
ORD000038|CUST017 |PRD011 |PLACED |2026-10-06 16:30:26|Customer ID not found in Silver; Product ID not found in Silver|
ORD000042|CUST018 |PRD011 |DELIVERED |2026-10-17 20:14:35|Product ID not found in Silver; Invalid or missing quantity |
ORD000045|CUST003 |NULL |DELIVERED |2026-11-06 19:26:50|Missing product ID; Product ID not found in Silver |
ORD000046|CUST017 |PRD005 |DELIVERED |2026-10-09 13:34:46|Customer ID not found in Silver |
ORD000049|CUST036 |PRD009 |DELIVERED |2026-09-19 19:24:21|Product ID not found in Silver |
ORD000054|CUST024 |PRD010 |DLVRD |2026-09-01 21:23:04|Unrecognized order status |
ORD000055|CUST009 |PRD004 |CANCELLED |2026-09-20 17:39:15|Product ID not found in Silver |
ORD000057|CUST031 |PRD017 |UNKNOWN |2026-10-09 18:07:22|Unrecognized order status |
ORD000061|CUST013 |PRD016 |CANCELLED |2026-11-03 14:20:11|Product ID not found in Silver |
ORD000064|CUST029 |PRD009 |SHIPPED |2026-09-16 13:17:53|Product ID not found in Silver |
ORD000065|CUST036 |PRD004 |DELIVERED |2026-10-05 11:21:09|Product ID not found in Silver |
ORD000066|CUST026 |PRD001 |CANCELLED |2026-11-13 15:40:25|Customer ID not found in Silver |
ORD000071|CUST008 |PRD024 |DELIVERED |2026-09-02 08:48:14|Customer ID not found in Silver; Missing or invalid order date |
ORD000072|CUST030 |PRD018 |PLACED |2026-09-03 20:13:50|Future order date |
ORD000073|CUST013 |PRD019 |DELIVERED |2026-10-18 09:35:44|Missing or invalid order date |
ORD000075|CUST034 |PRD024 |SHIPPED |2026-10-10 09:36:26|Customer ID not found in Silver |
ORD000077|CUST022 |PRD011 |DELIVERED |2026-11-05 11:58:30|Product ID not found in Silver |
ORD000078|CUST021 |PRD009 |CANCELLED |2026-11-17 15:16:31|Product ID not found in Silver |
ORD000079|CUST024 |PRD009 |DELIVERED |2026-11-02 10:01:30|Product ID not found in Silver |

Create the final Silver Orders DataFrame

In [0]:
%scala
val silverOrdersFinal = silverOrdersCandidates.select(
  col("order_id"),
  col("customer_id"),
  col("product_id"),
  col("quantity_parsed").cast("int").as("quantity"),
  col("order_date_parsed").cast("date").as("order_date"),
  col("order_status"),
  col("last_updated_parsed").cast("timestamp").as("last_updated"),
  col("_source_file"),
  col("_ingestion_timestamp")
)

println("========== FINAL SILVER ORDERS ==========")

println("Total Silver Orders: " + silverOrdersFinal.count())

silverOrdersFinal.printSchema()

silverOrdersFinal.orderBy("order_id").show(20, false)

========== FINAL SILVER ORDERS ==========
Total Silver Orders: 90
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: integer (nullable = true)
-- order_date: date (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: timestamp (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

+---------+-----------+----------+--------+----------+------------+-------------------+-------------------------------------------------------------------------------------------------+--------------------------+
order_id |customer_id|product_id|quantity|order_date|order_status|last_updated |_source_file |_ingestion_timestamp |
+---------+-----------+----------+--------+----------+------------+-------------------+-------------------------------------------------------------------------------------------------+--------------------------+
ORD000001|CUST023 |PRD005 |1 |2026-09-06|DELIVERED |2026-09-06 20:34:17|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000002|CUST032 |PRD005 |5 |2026-11-14|DELIVERED |2026-11-14 17:49:59|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000003|CUST033 |PRD010 |1 |2026-10-31|DELIVERED |2026-10-31 17:08:38|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000005|CUST013 |PRD013 |1 |2026-10-20|DELIVERED |2026-10-20 20:09:33|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000007|CUST021 |PRD024 |3 |2026-09-04|DELIVERED |2026-09-04 08:32:30|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000008|CUST030 |PRD020 |5 |2026-10-12|DELIVERED |2026-10-12 22:07:31|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000010|CUST005 |PRD019 |4 |2026-11-16|DELIVERED |2026-11-16 11:43:45|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000011|CUST003 |PRD022 |5 |2026-09-26|DELIVERED |2026-09-26 18:58:11|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000012|CUST013 |PRD017 |1 |2026-10-28|DELIVERED |2026-10-28 21:48:41|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000014|CUST035 |PRD015 |1 |2026-09-24|DELIVERED |2026-09-24 14:17:27|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000016|CUST021 |PRD018 |4 |2026-10-24|DELIVERED |2026-10-24 12:29:11|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000019|CUST011 |PRD012 |5 |2026-09-24|CANCELLED |2026-09-24 08:22:54|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000021|CUST019 |PRD020 |3 |2026-11-12|CANCELLED |2026-11-12 19:13:36|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000024|CUST005 |PRD010 |3 |2026-10-11|DELIVERED |2026-10-11 12:37:53|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000028|CUST021 |PRD021 |2 |2026-11-18|PLACED |2026-11-18 08:59:25|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000032|CUST022 |PRD007 |5 |2026-11-14

Write Silver Orders to Delta Lake

In [0]:
%scala
val silverOrdersPath = basePath + "silver/orders"

silverOrdersFinal.write
  .format("delta")
  .mode("overwrite")
  .save(silverOrdersPath)

println("Silver Orders Delta write successful")
println("Location: " + silverOrdersPath)

Silver Orders Delta write successful
Location: abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/orders
val silverOrdersPath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/orders

Verify the saved Delta table

In [0]:
%scala
val silverOrdersCheck = spark.read
  .format("delta")
  .load(silverOrdersPath)

println("========== SILVER ORDERS VERIFICATION ==========")

println("Saved Orders Count: " + silverOrdersCheck.count())

silverOrdersCheck.printSchema()

silverOrdersCheck
  .orderBy("order_id")
  .show(10, false)

========== SILVER ORDERS VERIFICATION ==========
Saved Orders Count: 90
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: integer (nullable = true)
-- order_date: date (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: timestamp (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

+---------+-----------+----------+--------+----------+------------+-------------------+-------------------------------------------------------------------------------------------------+--------------------------+
order_id |customer_id|product_id|quantity|order_date|order_status|last_updated |_source_file |_ingestion_timestamp |
+---------+-----------+----------+--------+----------+------------+-------------------+-------------------------------------------------------------------------------------------------+--------------------------+
ORD000001|CUST023 |PRD005 |1 |2026-09-06|DELIVERED |2026-09-06 20:34:17|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000002|CUST032 |PRD005 |5 |2026-11-14|DELIVERED |2026-11-14 17:49:59|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000003|CUST033 |PRD010 |1 |2026-10-31|DELIVERED |2026-10-31 17:08:38|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000005|CUST013 |PRD013 |1 |2026-10-20|DELIVERED |2026-10-20 20:09:33|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000007|CUST021 |PRD024 |3 |2026-09-04|DELIVERED |2026-09-04 08:32:30|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000008|CUST030 |PRD020 |5 |2026-10-12|DELIVERED |2026-10-12 22:07:31|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000010|CUST005 |PRD019 |4 |2026-11-16|DELIVERED |2026-11-16 11:43:45|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000011|CUST003 |PRD022 |5 |2026-09-26|DELIVERED |2026-09-26 18:58:11|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000012|CUST013 |PRD017 |1 |2026-10-28|DELIVERED |2026-10-28 21:48:41|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
ORD000014|CUST035 |PRD015 |1 |2026-09-24|DELIVERED |2026-09-24 14:17:27|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv|2026-11-20 17:21:45.893472|
+---------+-----------+----------+--------+----------+------------+-------------------+-------------------------------------------------------------------------------------------------+--------------------------+
only showing top 10 rows
val silverOrdersCheck: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 7 more fields]

Save rejected latest orders

In [0]:
%scala
val rejectedOrdersFinal = rejectedLatestOrders
  .drop("_valid_customer_id", "_valid_product_id")
  .select(
    col("order_id"),
    col("customer_id"),
    col("product_id"),
    col("quantity"),
    col("order_date"),
    col("order_status"),
    col("last_updated"),
    col("rejection_reason"),
    col("_source_file"),
    col("_ingestion_timestamp")
  )

val rejectedOrdersPath = basePath + "rejected/orders"

rejectedOrdersFinal.write
  .format("delta")
  .mode("overwrite")
  .save(rejectedOrdersPath)

println("Rejected Orders Delta write successful")
println("Rejected Orders Count: " + rejectedOrdersFinal.count())

Rejected Orders Delta write successful
Rejected Orders Count: 60
val rejectedOrdersFinal: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 8 more fields]
val rejectedOrdersPath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/rejected/orders

Verify both Delta tables and check for missing timestamps

In [0]:
%scala
val silverOrdersCheck = spark.read
  .format("delta")
  .load(basePath + "silver/orders")

val rejectedOrdersCheck = spark.read
  .format("delta")
  .load(basePath + "rejected/orders")

println("========== ORDERS PIPELINE VERIFICATION ==========")

println("Bronze Orders       : " + bronzeOrdersV2.count())
println("Silver Orders       : " + silverOrdersCheck.count())
println("Rejected Orders     : " + rejectedOrdersCheck.count())

println("========== INVALID UPDATE TIMESTAMP CHECK ==========")

val invalidUpdateTimestampOrders = distinctOrdersV1
  .filter(col("last_updated_parsed").isNull)

println(
  "Orders with missing/invalid last_updated: " +
  invalidUpdateTimestampOrders.count()
)

========== ORDERS PIPELINE VERIFICATION ==========
Bronze Orders : 161
Silver Orders : 90
Rejected Orders : 60
========== INVALID UPDATE TIMESTAMP CHECK ==========
Orders with missing/invalid last_updated: 0
val silverOrdersCheck: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 7 more fields]
val rejectedOrdersCheck: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 8 more fields]
val invalidUpdateTimestampOrders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 10 more fields]

Read the new CSV in Databricks

In [0]:
%scala

val ordersV2Path =
  basePath + "schema_evolution/orders_v2/orders_v2.csv"

println(ordersV2Path)

dbutils.fs.ls(
  basePath + "schema_evolution/orders_v2/"
).foreach(println)

basePath + &quot;schema_evolution/orders_v2/orders_v2.csv&quot;
 ^
On line 7: error: not found: value basePath
 basePath + &quot;schema_evolution/orders_v2/&quot;
 ^
On line 12: error: not found: value basePath
 ).foreach(println)
 ^
On line 13: warning: Auto-application to `()` is deprecated. Supply the empty argument list `()` explicitly to invoke method println,
 or remove the empty argument list from its definition (Java-defined methods are exempt).
 In Scala 3, an unapplied method like this will be eta-expanded into a function. [quickfixable]

In [0]:
%scala

import org.apache.spark.sql.types._

val ordersV2Schema = StructType(Seq(
  StructField("order_id", StringType, true),
  StructField("customer_id", StringType, true),
  StructField("product_id", StringType, true),
  StructField("quantity", StringType, true),
  StructField("order_date", StringType, true),
  StructField("order_status", StringType, true),
  StructField("last_updated", StringType, true),
  StructField("sales_channel", StringType, true),
  StructField("discount_amount", StringType, true)
))

println("Updated Orders V2 schema created successfully")
ordersV2Schema.printTreeString()

Updated Orders V2 schema created successfully
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: string (nullable = true)
-- order_date: string (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: string (nullable = true)

import org.apache.spark.sql.types._
val ordersV2Schema: org.apache.spark.sql.types.StructType = StructType(StructField(order_id,StringType,true),StructField(customer_id,StringType,true),StructField(product_id,StringType,true),StructField(quantity,StringType,true),StructField(order_date,StringType,true),StructField(order_status,StringType,true),StructField(last_updated,StringType,true),StructField(sales_channel,StringType,true),StructField(discount_amount,StringType,true))

In [0]:
%scala

val ordersV2Path =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/schema_evolution/orders_v2/orders_v2.csv"

val ordersV2Raw = spark.read
  .format("csv")
  .option("header", "true")
  .option("mode", "PERMISSIVE")
  .schema(ordersV2Schema)
  .load(ordersV2Path)

println("Orders V2 DataFrame created successfully")

Orders V2 DataFrame created successfully
val ordersV2Path: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/schema_evolution/orders_v2/orders_v2.csv
val ordersV2Raw: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 7 more fields]

In [0]:
%scala

println("========== ORDERS V2 VERIFICATION ==========")

println("Orders V2 Count: " + ordersV2Raw.count())

ordersV2Raw.printSchema()

ordersV2Raw.show(20, false)

========== ORDERS V2 VERIFICATION ==========
Orders V2 Count: 12
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: string (nullable = true)
-- order_date: string (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: string (nullable = true)

+---------+-----------+----------+--------+----------+------------+-------------------+-------------+---------------+
order_id |customer_id|product_id|quantity|order_date|order_status|last_updated |sales_channel|discount_amount|
+---------+-----------+----------+--------+----------+------------+-------------------+-------------+---------------+
ORD000159|CUST040 |PRD019 |4 |2026-11-17|PLACED |2026-11-17 09:43:00|STORE |150.00 |
ORD000160|CUST019 |PRD005 |1 |2026-11-15|PLACED |2026-11-15 14:14:00|WEB |0.00 |
ORD000161|CUST005 |PRD020 |4 |2026-11-15|PLACED |2026-11-15 08:08:00|MOBILE_APP |150.00 |
ORD000162|CUST019 |PRD015 |3 |2026-11-15|PLACED |2026-11-15 16:43:00|MOBILE_APP |0.00 |
ORD000163|CUST027 |PRD010 |2 |2026-11-15|PLACED |2026-11-15 12:40:00|STORE |0.00 |
ORD000164|CUST004 |PRD003 |4 |2026-11-18|PLACED |2026-11-18 17:26:00|MOBILE_APP |100.00 |
ORD000165|CUST001 |PRD010 |1 |2026-11-16|PLACED |2026-11-16 20:08:00|MOBILE_APP |0.00 |
ORD000166|CUST030 |PRD018 |2 |2026-11-17|PLACED |2026-11-17 11:20:00|WEB |50.00 |
ORD000167|CUST007 |PRD007 |4 |2026-11-15|PLACED |2026-11-15 14:04:00|MOBILE_APP |100.00 |
ORD000168|CUST001 |PRD012 |1 |2026-11-15|PLACED |2026-11-15 12:35:00|WEB |0.00 |
ORD000169|CUST017 |PRD003 |3 |2026-11-17|PLACED |2026-11-17 16:35:00|WEB |150.00 |
ORD000170|CUST015 |PRD016 |3 |2026-11-15|PLACED |2026-11-15 20:50:00|WEB |0.00 |
+---------+-----------+----------+--------+----------+------------+-------------------+-------------+---------------+

In [0]:
%scala
val basePath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/"
val existingBronzeOrders = spark.read
  .format("delta")
  .load(basePath + "bronze_delta/orders")

println("Existing Bronze DataFrame loaded successfully")

Existing Bronze DataFrame loaded successfully
val basePath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/
val existingBronzeOrders: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 7 more fields]

In [0]:
%scala

existingBronzeOrders.printSchema()

root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: string (nullable = true)
-- order_date: string (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

In [0]:
%scala

println(existingBronzeOrders.count())

161

Check overlapping order IDs

In [0]:
%scala

val overlappingOrders = ordersV2Raw
  .select("order_id")
  .distinct()
  .join(
    existingBronzeOrders.select("order_id").distinct(),
    Seq("order_id"),
    "inner"
  )

println("Overlapping Order IDs: " + overlappingOrders.count())
overlappingOrders.show(false)

Overlapping Order IDs: 0
+--------+
order_id|
+--------+
+--------+

val overlappingOrders: org.apache.spark.sql.DataFrame = [order_id: string]

Prepare V2 records for Bronze

In [0]:
%scala

import org.apache.spark.sql.functions._

val ordersV2ForBronze = ordersV2Raw
  .withColumn("_source_file", input_file_name())
  .withColumn("_ingestion_timestamp", current_timestamp())

println("Prepared V2 records: " + ordersV2ForBronze.count())
ordersV2ForBronze.printSchema()

Prepared V2 records: 12
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: string (nullable = true)
-- order_date: string (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: string (nullable = true)
-- _source_file: string (nullable = false)
-- _ingestion_timestamp: timestamp (nullable = false)

import org.apache.spark.sql.functions._
val ordersV2ForBronze: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 9 more fields]

Append V2 records with mergeSchema

In [0]:
%scala

ordersV2ForBronze.write
  .format("delta")
  .mode("append")
  .option("mergeSchema", "true")
  .save(basePath + "bronze_delta/orders")

println("Orders V2 appended successfully with schema evolution")

Orders V2 appended successfully with schema evolution

Reload the updated Bronze table

In [0]:
%scala

val bronzeOrdersEvolved = spark.read
  .format("delta")
  .load(basePath + "bronze_delta/orders")

println("Updated Bronze Orders loaded successfully")

Updated Bronze Orders loaded successfully
val bronzeOrdersEvolved: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 9 more fields]

Verify record count

In [0]:
%scala

println("Total Bronze Orders: " + bronzeOrdersEvolved.count())

Total Bronze Orders: 173

Verify evolved schema

In [0]:
%scala

bronzeOrdersEvolved.printSchema()

root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: string (nullable = true)
-- order_date: string (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: string (nullable = true)

Verify old records have NULL values

In [0]:
%scala

val oldOrdersNullCheck = bronzeOrdersEvolved
  .filter(col("sales_channel").isNull &&
          col("discount_amount").isNull)

println(
  "Records with NULL sales_channel and discount_amount: " +
  oldOrdersNullCheck.count()
)

Records with NULL sales_channel and discount_amount: 161
val oldOrdersNullCheck: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 9 more fields]

In [0]:
%scala

bronzeOrdersEvolved
  .filter(col("sales_channel").isNotNull)
  .select(
    "order_id",
    "customer_id",
    "product_id",
    "sales_channel",
    "discount_amount"
  )
  .orderBy("order_id")
  .show(20, false)

+---------+-----------+----------+-------------+---------------+
order_id |customer_id|product_id|sales_channel|discount_amount|
+---------+-----------+----------+-------------+---------------+
ORD000159|CUST040 |PRD019 |STORE |150.00 |
ORD000160|CUST019 |PRD005 |WEB |0.00 |
ORD000161|CUST005 |PRD020 |MOBILE_APP |150.00 |
ORD000162|CUST019 |PRD015 |MOBILE_APP |0.00 |
ORD000163|CUST027 |PRD010 |STORE |0.00 |
ORD000164|CUST004 |PRD003 |MOBILE_APP |100.00 |
ORD000165|CUST001 |PRD010 |MOBILE_APP |0.00 |
ORD000166|CUST030 |PRD018 |WEB |50.00 |
ORD000167|CUST007 |PRD007 |MOBILE_APP |100.00 |
ORD000168|CUST001 |PRD012 |WEB |0.00 |
ORD000169|CUST017 |PRD003 |WEB |150.00 |
ORD000170|CUST015 |PRD016 |WEB |0.00 |
+---------+-----------+----------+-------------+---------------+

Rebuild the Orders cleaning stage

In [0]:
%scala

val cleanedOrdersEvolved = bronzeOrdersEvolved
  .withColumn("order_id", upper(trim(col("order_id"))))
  .withColumn("customer_id", upper(trim(col("customer_id"))))
  .withColumn("product_id", upper(trim(col("product_id"))))
  .withColumn("order_status", upper(trim(col("order_status"))))
  .withColumn(
    "sales_channel",
    when(
      col("sales_channel").isNotNull,
      upper(trim(col("sales_channel")))
    ).otherwise(lit(null).cast("string"))
  )
  .withColumn(
    "quantity_parsed",
    expr("try_cast(trim(quantity) AS INT)")
  )
  .withColumn(
    "discount_amount_parsed",
    expr("try_cast(trim(discount_amount) AS DECIMAL(10,2))")
  )
  .withColumn(
    "order_date_parsed",
    coalesce(
      to_date(try_to_timestamp(col("order_date"), lit("yyyy-MM-dd"))),
      to_date(try_to_timestamp(col("order_date"), lit("dd-MM-yyyy")))
    )
  )
  .withColumn(
    "last_updated_parsed",
    try_to_timestamp(col("last_updated"), lit("yyyy-MM-dd HH:mm:ss"))
  )

println("Evolved Orders cleaning completed")

Evolved Orders cleaning completed
val cleanedOrdersEvolved: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 13 more fields]

Inspect the parsed V2 records

In [0]:
%scala

cleanedOrdersEvolved
  .filter(col("sales_channel").isNotNull)
  .select(
    "order_id",
    "quantity_parsed",
    "order_date_parsed",
    "order_status",
    "sales_channel",
    "discount_amount_parsed",
    "last_updated_parsed"
  )
  .orderBy("order_id")
  .show(20, false)

+---------+---------------+-----------------+------------+-------------+----------------------+-------------------+
order_id |quantity_parsed|order_date_parsed|order_status|sales_channel|discount_amount_parsed|last_updated_parsed|
+---------+---------------+-----------------+------------+-------------+----------------------+-------------------+
ORD000159|4 |2026-11-17 |PLACED |STORE |150.00 |2026-11-17 09:43:00|
ORD000160|1 |2026-11-15 |PLACED |WEB |0.00 |2026-11-15 14:14:00|
ORD000161|4 |2026-11-15 |PLACED |MOBILE_APP |150.00 |2026-11-15 08:08:00|
ORD000162|3 |2026-11-15 |PLACED |MOBILE_APP |0.00 |2026-11-15 16:43:00|
ORD000163|2 |2026-11-15 |PLACED |STORE |0.00 |2026-11-15 12:40:00|
ORD000164|4 |2026-11-18 |PLACED |MOBILE_APP |100.00 |2026-11-18 17:26:00|
ORD000165|1 |2026-11-16 |PLACED |MOBILE_APP |0.00 |2026-11-16 20:08:00|
ORD000166|2 |2026-11-17 |PLACED |WEB |50.00 |2026-11-17 11:20:00|
ORD000167|4 |2026-11-15 |PLACED |MOBILE_APP |100.00 |2026-11-15 14:04:00|
ORD000168|1 |2026-11-15 |PLACED |WEB |0.00 |2026-11-15 12:35:00|
ORD000169|3 |2026-11-17 |PLACED |WEB |150.00 |2026-11-17 16:35:00|
ORD000170|3 |2026-11-15 |PLACED |WEB |0.00 |2026-11-15 20:50:00|
+---------+---------------+-----------------+------------+-------------+----------------------+-------------------+

Prepare reference data

In [0]:
%scala

val silverCustomersRefEvolved = spark.read
  .format("delta")
  .load(basePath + "silver/customers")
  .select("customer_id")
  .distinct()

val silverProductsRefEvolved = spark.read
  .format("delta")
  .load(basePath + "silver/products")
  .select("product_id")
  .distinct()

println("Customer reference count: " + silverCustomersRefEvolved.count())
println("Product reference count: " + silverProductsRefEvolved.count())

Customer reference count: 36
Product reference count: 20
val silverCustomersRefEvolved: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: string]
val silverProductsRefEvolved: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product_id: string]

Check valid sales channels

In [0]:
%scala

cleanedOrdersEvolved
  .filter(
    col("sales_channel").isNotNull &&
    !col("sales_channel").isin("STORE", "WEB", "MOBILE_APP")
  )
  .select("order_id", "sales_channel")
  .show(false)

+--------+-------------+
order_id|sales_channel|
+--------+-------------+
+--------+-------------+

Build the complete Orders validation logic

In [0]:
%scala

val ordersValidatedEvolved = cleanedOrdersEvolved
  .join(
    silverCustomersRefEvolved
      .withColumn("_valid_customer_id", lit(true)),
    Seq("customer_id"),
    "left"
  )
  .join(
    silverProductsRefEvolved
      .withColumn("_valid_product_id", lit(true)),
    Seq("product_id"),
    "left"
  )
  .withColumn(
    "rejection_reason",
    concat_ws(
      "; ",
      when(
        col("order_id").isNull ||
        !col("order_id").rlike("^ORD[0-9]{6}$"),
        lit("Invalid order_id")
      ),
      when(
        col("customer_id").isNull ||
        col("_valid_customer_id").isNull,
        lit("Invalid or unknown customer_id")
      ),
      when(
        col("product_id").isNull ||
        col("_valid_product_id").isNull,
        lit("Invalid or unknown product_id")
      ),
      when(
        col("quantity_parsed").isNull ||
        col("quantity_parsed") <= 0,
        lit("Invalid quantity")
      ),
      when(
        col("order_date_parsed").isNull ||
        col("order_date_parsed") > current_date(),
        lit("Invalid or future order_date")
      ),
      when(
        col("last_updated_parsed").isNull,
        lit("Invalid last_updated")
      ),
      when(
        col("order_status").isNull ||
        !col("order_status").isin(
          "PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"
        ),
        lit("Invalid order_status")
      ),
      when(
        col("sales_channel").isNotNull &&
        !col("sales_channel").isin("STORE", "WEB", "MOBILE_APP"),
        lit("Invalid sales_channel")
      ),
      when(
        col("discount_amount").isNotNull &&
        (
          col("discount_amount_parsed").isNull ||
          col("discount_amount_parsed") < 0
        ),
        lit("Invalid discount_amount")
      )
    )
  )

println("Orders validation rules applied successfully")

Orders validation rules applied successfully
val ordersValidatedEvolved: org.apache.spark.sql.DataFrame = [product_id: string, customer_id: string ... 16 more fields]

Check validation counts

In [0]:
%scala

println(
  "Total records: " + ordersValidatedEvolved.count()
)

println(
  "Valid records before deduplication: " +
  ordersValidatedEvolved.filter(col("rejection_reason") === "").count()
)

println(
  "Rejected records before deduplication: " +
  ordersValidatedEvolved.filter(col("rejection_reason") =!= "").count()
)

Total records: 173
Valid records before deduplication: 109
Rejected records before deduplication: 64

Remove exact duplicate records

In [0]:
%scala

val distinctOrdersEvolved = ordersValidatedEvolved
  .dropDuplicates(Seq(
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "order_date",
    "order_status",
    "last_updated",
    "sales_channel",
    "discount_amount"
  ))

println(
  "Records after removing exact duplicates: " +
  distinctOrdersEvolved.count()
)

Records after removing exact duplicates: 168
val distinctOrdersEvolved: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product_id: string, customer_id: string ... 16 more fields]

Select the latest version of each order

In [0]:
%scala

import org.apache.spark.sql.expressions.Window

val orderLatestWindowEvolved = Window
  .partitionBy("order_id")

val latestOrderCandidatesEvolved = distinctOrdersEvolved
  .withColumn(
    "max_last_updated",
    max(col("last_updated_parsed")).over(orderLatestWindowEvolved)
  )
  .filter(
    col("last_updated_parsed") === col("max_last_updated")
  )
  .drop("max_last_updated")

println(
  "Latest order candidates: " +
  latestOrderCandidatesEvolved.count()
)

Latest order candidates: 162
import org.apache.spark.sql.expressions.Window
val orderLatestWindowEvolved: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@525f9a8
val latestOrderCandidatesEvolved: org.apache.spark.sql.DataFrame = [product_id: string, customer_id: string ... 16 more fields]

In [0]:
%scala

val latestTimestampConflictsEvolved = latestOrderCandidatesEvolved
  .groupBy("order_id")
  .count()
  .filter(col("count") > 1)

println(
  "Latest timestamp conflicts: " +
  latestTimestampConflictsEvolved.count()
)

latestTimestampConflictsEvolved.show(false)

Latest timestamp conflicts: 0
+--------+-----+
order_id|count|
+--------+-----+
+--------+-----+

val latestTimestampConflictsEvolved: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, count: bigint]

Separate valid and rejected latest orders

In [0]:
%scala

val silverOrdersEvolved = latestOrderCandidatesEvolved
  .filter(col("rejection_reason") === "")

val rejectedOrdersEvolved = latestOrderCandidatesEvolved
  .filter(col("rejection_reason") =!= "")

println(
  "Latest valid Orders: " + silverOrdersEvolved.count()
)

println(
  "Latest rejected Orders: " + rejectedOrdersEvolved.count()
)

Latest valid Orders: 100
Latest rejected Orders: 62
val silverOrdersEvolved: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product_id: string, customer_id: string ... 16 more fields]
val rejectedOrdersEvolved: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product_id: string, customer_id: string ... 16 more fields]

In [0]:
%scala

rejectedOrdersEvolved
  .filter(col("order_id").between("ORD000159", "ORD000170"))
  .select(
    "order_id",
    "customer_id",
    "product_id",
    "sales_channel",
    "discount_amount",
    "rejection_reason"
  )
  .orderBy("order_id")
  .show(false)

+---------+-----------+----------+-------------+---------------+------------------------------+
order_id |customer_id|product_id|sales_channel|discount_amount|rejection_reason |
+---------+-----------+----------+-------------+---------------+------------------------------+
ORD000169|CUST017 |PRD003 |WEB |150.00 |Invalid or unknown customer_id|
ORD000170|CUST015 |PRD016 |WEB |0.00 |Invalid or unknown product_id |
+---------+-----------+----------+-------------+---------------+------------------------------+

Prepare final Silver Orders schema

In [0]:
%scala

val silverOrdersFinalEvolved = silverOrdersEvolved.select(
  col("order_id"),
  col("customer_id"),
  col("product_id"),
  col("quantity_parsed").cast("int").as("quantity"),
  col("order_date_parsed").cast("date").as("order_date"),
  col("order_status"),
  col("sales_channel"),
  col("discount_amount_parsed")
    .cast("decimal(10,2)")
    .as("discount_amount"),
  col("last_updated_parsed").cast("timestamp").as("last_updated"),
  col("_source_file"),
  col("_ingestion_timestamp")
)

println("Final Silver Orders prepared")
silverOrdersFinalEvolved.printSchema()

Final Silver Orders prepared
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: integer (nullable = true)
-- order_date: date (nullable = true)
-- order_status: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: decimal(10,2) (nullable = true)
-- last_updated: timestamp (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

val silverOrdersFinalEvolved: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 9 more fields]

Prepare final Rejected Orders

In [0]:
%scala

val rejectedOrdersFinalEvolved = rejectedOrdersEvolved
  .select(
    col("order_id"),
    col("customer_id"),
    col("product_id"),
    col("quantity"),
    col("order_date"),
    col("order_status"),
    col("last_updated"),
    col("sales_channel"),
    col("discount_amount"),
    col("rejection_reason"),
    col("_source_file"),
    col("_ingestion_timestamp")
  )

println("Final Rejected Orders prepared")
rejectedOrdersFinalEvolved.printSchema()

Final Rejected Orders prepared
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: string (nullable = true)
-- order_date: string (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: string (nullable = true)
-- rejection_reason: string (nullable = false)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

val rejectedOrdersFinalEvolved: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]

Verify final counts

In [0]:
%scala

println("Bronze Orders: " + bronzeOrdersEvolved.count())
println("Silver Orders: " + silverOrdersFinalEvolved.count())
println("Rejected Orders: " + rejectedOrdersFinalEvolved.count())

Bronze Orders: 173
Silver Orders: 100
Rejected Orders: 62

Write Silver Orders to ADLS Gen2

In [0]:
%scala

silverOrdersFinalEvolved.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .save(basePath + "silver/orders")

println("Silver Orders updated successfully")

Silver Orders updated successfully

Write Rejected Orders to ADLS Gen2

In [0]:
%scala

rejectedOrdersFinalEvolved.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .save(basePath + "rejected/orders")

println("Rejected Orders updated successfully")

Rejected Orders updated successfully

Read back and verify persisted Delta tables

In [0]:
%scala

val silverOrdersVerified = spark.read
  .format("delta")
  .load(basePath + "silver/orders")

val rejectedOrdersVerified = spark.read
  .format("delta")
  .load(basePath + "rejected/orders")

println("========== ORDERS PIPELINE VERIFICATION ==========")

println("Silver Orders: " + silverOrdersVerified.count())
println("Rejected Orders: " + rejectedOrdersVerified.count())

println("========== SILVER SCHEMA ==========")
silverOrdersVerified.printSchema()

println("========== REJECTED SCHEMA ==========")
rejectedOrdersVerified.printSchema()

========== ORDERS PIPELINE VERIFICATION ==========
Silver Orders: 100
Rejected Orders: 62
========== SILVER SCHEMA ==========
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: integer (nullable = true)
-- order_date: date (nullable = true)
-- order_status: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: decimal(10,2) (nullable = true)
-- last_updated: timestamp (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

========== REJECTED SCHEMA ==========
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: string (nullable = true)
-- order_date: string (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: string (nullable = true)
-- rejection_reason: string (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

val silverOrdersVerified: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 9 more fields]
val rejectedOrdersVerified: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]

## Gold Star Schema

#### Load Silver source tables

In [0]:
%scala

val goldCustomersSource = spark.read
  .format("delta")
  .load(basePath + "silver/customers")

val goldProductsSource = spark.read
  .format("delta")
  .load(basePath + "silver/products")

val goldOrdersSource = spark.read
  .format("delta")
  .load(basePath + "silver/orders")

println("========== GOLD SOURCE VERIFICATION ==========")
println("Customers: " + goldCustomersSource.count())
println("Products: " + goldProductsSource.count())
println("Orders: " + goldOrdersSource.count())

========== GOLD SOURCE VERIFICATION ==========
Customers: 36
Products: 20
Orders: 100
val goldCustomersSource: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 5 more fields]
val goldProductsSource: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 5 more fields]
val goldOrdersSource: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 9 more fields]

#### Build DIM_CUSTOMER

In [0]:
%scala

import org.apache.spark.sql.expressions.Window

val customerKeyWindow = Window
  .orderBy(col("customer_id"))

val dimCustomer = goldCustomersSource
  .select(
    col("customer_id"),
    col("customer_name"),
    col("email"),
    col("city"),
    col("signup_date")
  )
  .withColumn(
    "customer_key",
    row_number().over(customerKeyWindow)
  )
  .select(
    col("customer_key"),
    col("customer_id"),
    col("customer_name"),
    col("email"),
    col("city"),
    col("signup_date")
  )

println("DIM_CUSTOMER prepared successfully")
println("DIM_CUSTOMER count: " + dimCustomer.count())

dimCustomer.printSchema()
dimCustomer.show(10, false)

DIM_CUSTOMER prepared successfully
DIM_CUSTOMER count: 36
root
-- customer_key: integer (nullable = false)
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)
-- email: string (nullable = true)
-- city: string (nullable = true)
-- signup_date: date (nullable = true)

+------------+-----------+-------------+-----------------------------+---------+-----------+
customer_key|customer_id|customer_name|email |city |signup_date|
+------------+-----------+-------------+-----------------------------+---------+-----------+
1 |CUST001 |Priya Patel |priya.patel1@mailexample.in |Mumbai |2025-06-24 |
2 |CUST002 |Sneha Gupta |sneha.gupta2@mailexample.in |Kolkata |2025-03-13 |
3 |CUST003 |Divya Singh |divya.singh3@mailexample.in |Kochi |2025-02-08 |
4 |CUST004 |Suresh Gupta |suresh.gupta4@mailexample.in |Kochi |2023-09-03 |
5 |CUST005 |Aarav Gupta |aarav.gupta5@mailexample.in |Mumbai |2023-04-24 |
6 |CUST006 |Vikram Sharma|vikram.sharma6@mailexample.in|Kochi |2023-01-12 |
7 |CUST007 |Lakshmi Rao |lakshmi.rao7@mailexample.in |Pune |2023-08-04 |
8 |CUST009 |Anita Das |anita.das9@mailexample.in |Mumbai |2025-01-10 |
9 |CUST010 |Anita Sharma |anita.sharma10@mailexample.in|Hyderabad|2023-10-22 |
10 |CUST011 |Meera Iyer |meera.iyer11@mailexample.in |Chennai |2023-01-26 |
+------------+-----------+-------------+-----------------------------+---------+-----------+
only showing top 10 rows
import org.apache.spark.sql.expressions.Window
val customerKeyWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@2277c253
val dimCustomer: org.apache.spark.sql.DataFrame = [customer_key: int, customer_id: string ... 4 more fields]

#### Build DIM_PRODUCT

In [0]:
%scala

val productKeyWindow = Window
  .orderBy(col("product_id"))

val dimProduct = goldProductsSource
  .select(
    col("product_id"),
    col("product_name"),
    col("category"),
    col("unit_price"),
    col("active")
  )
  .withColumn(
    "product_key",
    row_number().over(productKeyWindow)
  )
  .select(
    col("product_key"),
    col("product_id"),
    col("product_name"),
    col("category"),
    col("unit_price"),
    col("active")
  )

println("DIM_PRODUCT prepared successfully")
println("DIM_PRODUCT count: " + dimProduct.count())

dimProduct.printSchema()
dimProduct.show(10, false)

DIM_PRODUCT prepared successfully
DIM_PRODUCT count: 20
root
-- product_key: integer (nullable = false)
-- product_id: string (nullable = true)
-- product_name: string (nullable = true)
-- category: string (nullable = true)
-- unit_price: decimal(10,2) (nullable = true)
-- active: string (nullable = true)

+-----------+----------+-----------------+-----------+----------+------+
product_key|product_id|product_name |category |unit_price|active|
+-----------+----------+-----------------+-----------+----------+------+
1 |PRD001 |Wireless Earbuds |Electronics|2499.00 |Y |
2 |PRD002 |Smart Watch |Electronics|4999.00 |Y |
3 |PRD003 |Bluetooth Speaker|Electronics|1899.00 |Y |
4 |PRD005 |Cotton T-Shirt |Apparel |599.00 |Y |
5 |PRD006 |Denim Jeans |Apparel |1799.00 |Y |
6 |PRD007 |Running Shoes |Apparel |3299.00 |Y |
7 |PRD008 |Hoodie |Apparel |1499.00 |Y |
8 |PRD010 |Cookware Set |Home |2999.00 |Y |
9 |PRD012 |Bedsheet Set |Home |1299.00 |Y |
10 |PRD013 |Basmati Rice 5kg |Grocery |649.00 |Y |
+-----------+----------+-----------------+-----------+----------+------+
only showing top 10 rows
val productKeyWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@18413080
val dimProduct: org.apache.spark.sql.DataFrame = [product_key: int, product_id: string ... 4 more fields]

#### Build DIM_DATE

In [0]:
%scala

val orderDateRange = goldOrdersSource
  .agg(
    min(col("order_date")).as("min_date"),
    max(col("order_date")).as("max_date")
  )

orderDateRange.show(false)

+----------+----------+
min_date |max_date |
+----------+----------+
2026-09-01|2026-11-18|
+----------+----------+

val orderDateRange: org.apache.spark.sql.DataFrame = [min_date: date, max_date: date]

Generate DIM_DATE

In [0]:
%scala

val dimDate = orderDateRange
  .select(
    explode(
      sequence(
        col("min_date"),
        col("max_date"),
        expr("interval 1 day")
      )
    ).as("full_date")
  )
  .withColumn(
    "date_key",
    date_format(col("full_date"), "yyyyMMdd").cast("int")
  )
  .withColumn("day", dayofmonth(col("full_date")))
  .withColumn("month", month(col("full_date")))
  .withColumn("month_name", date_format(col("full_date"), "MMMM"))
  .withColumn("quarter", quarter(col("full_date")))
  .withColumn("year", year(col("full_date")))
  .withColumn("day_of_week", date_format(col("full_date"), "EEEE"))
  .withColumn(
    "is_weekend",
    dayofweek(col("full_date")).isin(1, 7)
  )
  .select(
    col("date_key"),
    col("full_date"),
    col("day"),
    col("month"),
    col("month_name"),
    col("quarter"),
    col("year"),
    col("day_of_week"),
    col("is_weekend")
  )

println("DIM_DATE prepared successfully")
println("DIM_DATE count: " + dimDate.count())

dimDate.printSchema()
dimDate.show(10, false)

DIM_DATE prepared successfully
DIM_DATE count: 79
root
-- date_key: integer (nullable = true)
-- full_date: date (nullable = false)
-- day: integer (nullable = false)
-- month: integer (nullable = false)
-- month_name: string (nullable = false)
-- quarter: integer (nullable = false)
-- year: integer (nullable = false)
-- day_of_week: string (nullable = false)
-- is_weekend: boolean (nullable = false)

+--------+----------+---+-----+----------+-------+----+-----------+----------+
date_key|full_date |day|month|month_name|quarter|year|day_of_week|is_weekend|
+--------+----------+---+-----+----------+-------+----+-----------+----------+
20260901|2026-09-01|1 |9 |September |3 |2026|Tuesday |false |
20260902|2026-09-02|2 |9 |September |3 |2026|Wednesday |false |
20260903|2026-09-03|3 |9 |September |3 |2026|Thursday |false |
20260904|2026-09-04|4 |9 |September |3 |2026|Friday |false |
20260905|2026-09-05|5 |9 |September |3 |2026|Saturday |true |
20260906|2026-09-06|6 |9 |September |3 |2026|Sunday |true |
20260907|2026-09-07|7 |9 |September |3 |2026|Monday |false |
20260908|2026-09-08|8 |9 |September |3 |2026|Tuesday |false |
20260909|2026-09-09|9 |9 |September |3 |2026|Wednesday |false |
20260910|2026-09-10|10 |9 |September |3 |2026|Thursday |false |
+--------+----------+---+-----+----------+-------+----+-----------+----------+
only showing top 10 rows
val dimDate: org.apache.spark.sql.DataFrame = [date_key: int, full_date: date ... 7 more fields]

#### Verify all three dimensions

In [0]:
%scala

println("========== DIMENSION COUNTS ==========")
println("DIM_CUSTOMER: " + dimCustomer.count())
println("DIM_PRODUCT : " + dimProduct.count())
println("DIM_DATE    : " + dimDate.count())

println("========== UNIQUE KEY CHECKS ==========")

println(
  "Duplicate customer keys: " +
  dimCustomer.groupBy("customer_key").count()
    .filter(col("count") > 1).count()
)

println(
  "Duplicate product keys: " +
  dimProduct.groupBy("product_key").count()
    .filter(col("count") > 1).count()
)

println(
  "Duplicate date keys: " +
  dimDate.groupBy("date_key").count()
    .filter(col("count") > 1).count()
)

========== DIMENSION COUNTS ==========
DIM_CUSTOMER: 36
DIM_PRODUCT : 20
DIM_DATE : 79
========== UNIQUE KEY CHECKS ==========
Duplicate customer keys: 0
Duplicate product keys: 0
Duplicate date keys: 0

#### Build FACT_ORDER

In [0]:
%scala

val ordersWithDimensionKeys = goldOrdersSource
  .join(
    dimCustomer.select("customer_id", "customer_key"),
    Seq("customer_id"),
    "inner"
  )
  .join(
    dimProduct.select("product_id", "product_key"),
    Seq("product_id"),
    "inner"
  )
  .join(
    dimDate.select(
      col("full_date").as("date_join"),
      col("date_key")
    ),
    col("order_date") === col("date_join"),
    "inner"
  )
  .drop("date_join")

println("Orders joined with dimensions: " + ordersWithDimensionKeys.count())

ordersWithDimensionKeys.select(
  "order_id",
  "customer_id",
  "customer_key",
  "product_id",
  "product_key",
  "order_date",
  "date_key"
).show(10, false)

Orders joined with dimensions: 100
+---------+-----------+------------+----------+-----------+----------+--------+
order_id |customer_id|customer_key|product_id|product_key|order_date|date_key|
+---------+-----------+------------+----------+-----------+----------+--------+
ORD000001|CUST023 |21 |PRD005 |4 |2026-09-06|20260906|
ORD000002|CUST032 |29 |PRD005 |4 |2026-11-14|20261114|
ORD000003|CUST033 |30 |PRD010 |8 |2026-10-31|20261031|
ORD000005|CUST013 |12 |PRD013 |10 |2026-10-20|20261020|
ORD000007|CUST021 |19 |PRD024 |20 |2026-09-04|20260904|
ORD000008|CUST030 |27 |PRD020 |16 |2026-10-12|20261012|
ORD000010|CUST005 |5 |PRD019 |15 |2026-11-16|20261116|
ORD000011|CUST003 |3 |PRD022 |18 |2026-09-26|20260926|
ORD000012|CUST013 |12 |PRD017 |13 |2026-10-28|20261028|
ORD000014|CUST035 |31 |PRD015 |12 |2026-09-24|20260924|
+---------+-----------+------------+----------+-----------+----------+--------+
only showing top 10 rows
val ordersWithDimensionKeys: org.apache.spark.sql.DataFrame = [product_id: string, customer_id: string ... 12 more fields]

##### Generate sales_key and finalize FACT_ORDER

In [0]:
%scala

val salesKeyWindow = Window
  .orderBy(col("order_id"))

val factOrder = ordersWithDimensionKeys
  .withColumn(
    "sales_key",
    row_number().over(salesKeyWindow)
  )
  .select(
    col("sales_key"),
    col("order_id"),
    col("customer_key"),
    col("product_key"),
    col("date_key"),
    col("quantity"),
    col("order_status"),
    col("sales_channel"),
    col("discount_amount"),
    col("last_updated")
  )

println("FACT_ORDER prepared successfully")
println("FACT_ORDER count: " + factOrder.count())

factOrder.printSchema()
factOrder.show(10, false)

FACT_ORDER prepared successfully
FACT_ORDER count: 100
root
-- sales_key: integer (nullable = false)
-- order_id: string (nullable = true)
-- customer_key: integer (nullable = false)
-- product_key: integer (nullable = false)
-- date_key: integer (nullable = true)
-- quantity: integer (nullable = true)
-- order_status: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: decimal(10,2) (nullable = true)
-- last_updated: timestamp (nullable = true)

+---------+---------+------------+-----------+--------+--------+------------+-------------+---------------+-------------------+
sales_key|order_id |customer_key|product_key|date_key|quantity|order_status|sales_channel|discount_amount|last_updated |
+---------+---------+------------+-----------+--------+--------+------------+-------------+---------------+-------------------+
1 |ORD000001|21 |4 |20260906|1 |DELIVERED |NULL |NULL |2026-09-06 20:34:17|
2 |ORD000002|29 |4 |20261114|5 |DELIVERED |NULL |NULL |2026-11-14 17:49:59|
3 |ORD000003|30 |8 |20261031|1 |DELIVERED |NULL |NULL |2026-10-31 17:08:38|
4 |ORD000005|12 |10 |20261020|1 |DELIVERED |NULL |NULL |2026-10-20 20:09:33|
5 |ORD000007|19 |20 |20260904|3 |DELIVERED |NULL |NULL |2026-09-04 08:32:30|
6 |ORD000008|27 |16 |20261012|5 |DELIVERED |NULL |NULL |2026-10-12 22:07:31|
7 |ORD000010|5 |15 |20261116|4 |DELIVERED |NULL |NULL |2026-11-16 11:43:45|
8 |ORD000011|3 |18 |20260926|5 |DELIVERED |NULL |NULL |2026-09-26 18:58:11|
9 |ORD000012|12 |13 |20261028|1 |DELIVERED |NULL |NULL |2026-10-28 21:48:41|
10 |ORD000014|31 |12 |20260924|1 |DELIVERED |NULL |NULL |2026-09-24 14:17:27|
+---------+---------+------------+-----------+--------+--------+------------+-------------+---------------+-------------------+
only showing top 10 rows
val salesKeyWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@45c82317
val factOrder: org.apache.spark.sql.DataFrame = [sales_key: int, order_id: string ... 8 more fields]

#### Validate FACT_ORDER

In [0]:
%scala
println("========== FACT_ORDER VALIDATION ==========")

println("Total Records: " + factOrder.count())

println(
  "Duplicate sales_key: " +
  factOrder.groupBy("sales_key")
    .count()
    .filter(col("count") > 1)
    .count()
)

println(
  "Null sales_key: " +
  factOrder.filter(col("sales_key").isNull).count()
)

println(
  "Null customer_key: " +
  factOrder.filter(col("customer_key").isNull).count()
)

println(
  "Null product_key: " +
  factOrder.filter(col("product_key").isNull).count()
)

println(
  "Null date_key: " +
  factOrder.filter(col("date_key").isNull).count()
)

========== FACT_ORDER VALIDATION ==========
Total Records: 100
Duplicate sales_key: 0
Null sales_key: 0
Null customer_key: 0
Null product_key: 0
Null date_key: 0

#### Validate foreign-key relationships

In [0]:
%scala
println("========== FOREIGN KEY VALIDATION ==========")

val invalidCustomerKeys = factOrder
  .join(
    dimCustomer.select("customer_key"),
    Seq("customer_key"),
    "left_anti"
  )

val invalidProductKeys = factOrder
  .join(
    dimProduct.select("product_key"),
    Seq("product_key"),
    "left_anti"
  )

val invalidDateKeys = factOrder
  .join(
    dimDate.select("date_key"),
    Seq("date_key"),
    "left_anti"
  )

println("Invalid customer keys: " + invalidCustomerKeys.count())
println("Invalid product keys: " + invalidProductKeys.count())
println("Invalid date keys: " + invalidDateKeys.count())

========== FOREIGN KEY VALIDATION ==========
Invalid customer keys: 0
Invalid product keys: 0
Invalid date keys: 0
val invalidCustomerKeys: org.apache.spark.sql.DataFrame = [customer_key: int, sales_key: int ... 8 more fields]
val invalidProductKeys: org.apache.spark.sql.DataFrame = [product_key: int, sales_key: int ... 8 more fields]
val invalidDateKeys: org.apache.spark.sql.DataFrame = [date_key: int, sales_key: int ... 8 more fields]

#### Write Gold tables to ADLS Gen2

In [0]:
%scala
// DIM_CUSTOMER
dimCustomer.write
  .format("delta")
  .mode("overwrite")
  .save(basePath + "gold/dim_customer")

// DIM_PRODUCT
dimProduct.write
  .format("delta")
  .mode("overwrite")
  .save(basePath + "gold/dim_product")

// DIM_DATE
dimDate.write
  .format("delta")
  .mode("overwrite")
  .save(basePath + "gold/dim_date")

// FACT_ORDER
factOrder.write
  .format("delta")
  .mode("overwrite")
  .save(basePath + "gold/fact_order")

println("========== GOLD LAYER WRITE SUMMARY ==========")
println("DIM_CUSTOMER written successfully")
println("DIM_PRODUCT written successfully")
println("DIM_DATE written successfully")
println("FACT_ORDER written successfully")

========== GOLD LAYER WRITE SUMMARY ==========
DIM_CUSTOMER written successfully
DIM_PRODUCT written successfully
DIM_DATE written successfully
FACT_ORDER written successfully

Reload and verify persisted Gold tables

In [0]:
%scala
val dimCustomerVerified = spark.read.format("delta")
  .load(basePath + "gold/dim_customer")

val dimProductVerified = spark.read.format("delta")
  .load(basePath + "gold/dim_product")

val dimDateVerified = spark.read.format("delta")
  .load(basePath + "gold/dim_date")

val factOrderVerified = spark.read.format("delta")
  .load(basePath + "gold/fact_order")

println("========== GOLD LAYER VERIFICATION ==========")

println("DIM_CUSTOMER: " + dimCustomerVerified.count())
println("DIM_PRODUCT: " + dimProductVerified.count())
println("DIM_DATE: " + dimDateVerified.count())
println("FACT_ORDER: " + factOrderVerified.count())

println("\n========== FACT_ORDER SCHEMA ==========")
factOrderVerified.printSchema()

println("\n========== DIM_CUSTOMER SCHEMA ==========")
dimCustomerVerified.printSchema()

println("\n========== DIM_PRODUCT SCHEMA ==========")
dimProductVerified.printSchema()

println("\n========== DIM_DATE SCHEMA ==========")
dimDateVerified.printSchema()

========== GOLD LAYER VERIFICATION ==========
DIM_CUSTOMER: 36
DIM_PRODUCT: 20
DIM_DATE: 79
FACT_ORDER: 100

========== FACT_ORDER SCHEMA ==========
root
-- sales_key: integer (nullable = true)
-- order_id: string (nullable = true)
-- customer_key: integer (nullable = true)
-- product_key: integer (nullable = true)
-- date_key: integer (nullable = true)
-- quantity: integer (nullable = true)
-- order_status: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: decimal(10,2) (nullable = true)
-- last_updated: timestamp (nullable = true)


========== DIM_CUSTOMER SCHEMA ==========
root
-- customer_key: integer (nullable = true)
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)
-- email: string (nullable = true)
-- city: string (nullable = true)
-- signup_date: date (nullable = true)


========== DIM_PRODUCT SCHEMA ==========
root
-- product_key: integer (nullable = true)
-- product_id: string (nullable = true)
-- product_name: string (nullable = true)
-- category: string (nullable = true)
-- unit_price: decimal(10,2) (nullable = true)
-- active: string (nullable = true)


========== DIM_DATE SCHEMA ==========
root
-- date_key: integer (nullable = true)
-- full_date: date (nullable = true)
-- day: integer (nullable = true)
-- month: integer (nullable = true)
-- month_name: string (nullable = true)
-- quarter: integer (nullable = true)
-- year: integer (nullable = true)
-- day_of_week: string (nullable = true)
-- is_weekend: boolean (nullable = true)

val dimCustomerVerified: org.apache.spark.sql.DataFrame = [customer_key: int, customer_id: string ... 4 more fields]
val dimProductVerified: org.apache.spark.sql.DataFrame = [product_key: int, product_id: string ... 4 more fields]
val dimDateVerified: org.apache.spark.sql.DataFrame = [date_key: int, full_date: date ... 7 more fields]
val factOrderVerified: org.apache.spark.sql.DataFrame = [sales_key: int, order_id: string ... 8 more fields]

## Check available catalogs

In [0]:
%sql
SHOW CATALOGS;

catalog
adlsdatabricks2_workspace
orders_demo
samples
system


In [0]:
%sql
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
adlsdatabricks2_workspace,default


#### Create the Gold schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adlsdatabricks2_workspace.gold;

In [0]:
%sql
SHOW SCHEMAS IN adlsdatabricks2_workspace;

databaseName
default
gold
information_schema


#### Register DIM_CUSTOMER

In [0]:
%sql
CREATE TABLE IF NOT EXISTS
adlsdatabricks2_workspace.gold.dim_customer
USING DELTA
LOCATION 'abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/gold/dim_customer';

In [0]:
%sql
SELECT * 
FROM adlsdatabricks2_workspace.gold.dim_customer
LIMIT 10;

customer_key,customer_id,customer_name,email,city,signup_date
1,CUST001,Priya Patel,priya.patel1@mailexample.in,Mumbai,2025-06-24
2,CUST002,Sneha Gupta,sneha.gupta2@mailexample.in,Kolkata,2025-03-13
3,CUST003,Divya Singh,divya.singh3@mailexample.in,Kochi,2025-02-08
4,CUST004,Suresh Gupta,suresh.gupta4@mailexample.in,Kochi,2023-09-03
5,CUST005,Aarav Gupta,aarav.gupta5@mailexample.in,Mumbai,2023-04-24
6,CUST006,Vikram Sharma,vikram.sharma6@mailexample.in,Kochi,2023-01-12
7,CUST007,Lakshmi Rao,lakshmi.rao7@mailexample.in,Pune,2023-08-04
8,CUST009,Anita Das,anita.das9@mailexample.in,Mumbai,2025-01-10
9,CUST010,Anita Sharma,anita.sharma10@mailexample.in,Hyderabad,2023-10-22
10,CUST011,Meera Iyer,meera.iyer11@mailexample.in,Chennai,2023-01-26


#### Register DIM_PRODUCT

In [0]:
%sql
CREATE TABLE IF NOT EXISTS
adlsdatabricks2_workspace.gold.dim_product
USING DELTA
LOCATION 'abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/gold/dim_product';

#### Register DIM_DATE

In [0]:
%sql
CREATE TABLE IF NOT EXISTS
adlsdatabricks2_workspace.gold.dim_date
USING DELTA
LOCATION 'abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/gold/dim_date';

#### Register FACT_ORDER

In [0]:
%sql
CREATE TABLE IF NOT EXISTS
adlsdatabricks2_workspace.gold.fact_order
USING DELTA
LOCATION 'abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/gold/fact_order';


#### Verify all catalog tables

In [0]:
%sql
SHOW TABLES IN adlsdatabricks2_workspace.gold;

database,tableName,isTemporary
gold,dim_customer,false
gold,dim_date,false
gold,dim_product,false
gold,fact_order,false


##### Verify catalog table counts

In [0]:
%sql
SELECT 'DIM_CUSTOMER' AS table_name, COUNT(*) AS total_records
FROM adlsdatabricks2_workspace.gold.dim_customer

UNION ALL

SELECT 'DIM_PRODUCT', COUNT(*)
FROM adlsdatabricks2_workspace.gold.dim_product

UNION ALL

SELECT 'DIM_DATE', COUNT(*)
FROM adlsdatabricks2_workspace.gold.dim_date

UNION ALL

SELECT 'FACT_ORDER', COUNT(*)
FROM adlsdatabricks2_workspace.gold.fact_order;

table_name,total_records
DIM_CUSTOMER,36
DIM_PRODUCT,20
DIM_DATE,79
FACT_ORDER,100


## Dashboard consumption layer

### order_status_summary

In [0]:
%sql
CREATE OR REPLACE VIEW
adlsdatabricks2_workspace.gold.vw_order_status_summary
AS
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM adlsdatabricks2_workspace.gold.fact_order
GROUP BY order_status;

In [0]:
%sql
SELECT *
FROM adlsdatabricks2_workspace.gold.vw_order_status_summary
ORDER BY total_orders DESC;

order_status,total_orders
DELIVERED,53
PLACED,18
CANCELLED,16
SHIPPED,9
RETURNED,4


#### Product category analysis

In [0]:
%sql
CREATE OR REPLACE VIEW
adlsdatabricks2_workspace.gold.vw_product_category_summary
AS
SELECT
    p.category,
    SUM(f.quantity) AS total_quantity,
    COUNT(DISTINCT f.order_id) AS total_orders
FROM adlsdatabricks2_workspace.gold.fact_order f
JOIN adlsdatabricks2_workspace.gold.dim_product p
    ON f.product_key = p.product_key
GROUP BY p.category;

In [0]:
%sql
SELECT *
FROM adlsdatabricks2_workspace.gold.vw_product_category_summary
ORDER BY total_quantity DESC;

category,total_quantity,total_orders
Apparel,87,27
Home,53,19
Beauty,49,14
Sports,46,16
Grocery,34,13
Electronics,28,11


#### Sales channel analysis

In [0]:
%sql
CREATE OR REPLACE VIEW
adlsdatabricks2_workspace.gold.vw_sales_channel_summary
AS
SELECT
    COALESCE(sales_channel, 'LEGACY/UNKNOWN') AS sales_channel,
    COUNT(*) AS total_orders,
    SUM(quantity) AS total_quantity
FROM adlsdatabricks2_workspace.gold.fact_order
GROUP BY COALESCE(sales_channel, 'LEGACY/UNKNOWN');

In [0]:
%sql
SELECT *
FROM adlsdatabricks2_workspace.gold.vw_sales_channel_summary
ORDER BY total_orders DESC;

sales_channel,total_orders,total_quantity
LEGACY/UNKNOWN,90,271
MOBILE_APP,5,16
WEB,3,4
STORE,2,6


#### Monthly order trend

In [0]:
%sql
CREATE OR REPLACE VIEW
adlsdatabricks2_workspace.gold.vw_monthly_order_trend
AS
SELECT
    d.year,
    d.month,
    d.month_name,
    COUNT(f.order_id) AS total_orders,
    SUM(f.quantity) AS total_quantity
FROM adlsdatabricks2_workspace.gold.fact_order f
JOIN adlsdatabricks2_workspace.gold.dim_date d
    ON f.date_key = d.date_key
GROUP BY
    d.year,
    d.month,
    d.month_name;

In [0]:
%sql
SELECT *
FROM adlsdatabricks2_workspace.gold.vw_monthly_order_trend
ORDER BY year, month;

year,month,month_name,total_orders,total_quantity
2026,9,September,29,86
2026,10,October,33,99
2026,11,November,38,112


#### Top 10 customers

In [0]:
%sql
CREATE OR REPLACE VIEW
adlsdatabricks2_workspace.gold.vw_top_customers
AS
SELECT
    c.customer_key,
    c.customer_id,
    c.customer_name,
    c.city,
    COUNT(DISTINCT f.order_id) AS total_orders,
    SUM(f.quantity) AS total_quantity
FROM adlsdatabricks2_workspace.gold.fact_order f
JOIN adlsdatabricks2_workspace.gold.dim_customer c
    ON f.customer_key = c.customer_key
GROUP BY
    c.customer_key,
    c.customer_id,
    c.customer_name,
    c.city
ORDER BY
    total_quantity DESC,
    total_orders DESC
LIMIT 10;

In [0]:
%sql
SELECT *
FROM adlsdatabricks2_workspace.gold.vw_top_customers
ORDER BY total_quantity DESC, total_orders DESC;

customer_key,customer_id,customer_name,city,total_orders,total_quantity
31,CUST035,Neha Rao,Mumbai,6,21
24,CUST027,Rohan Gupta,Kochi,6,19
29,CUST032,Meera Sharma,Kochi,4,17
19,CUST021,Meera Nair,Mumbai,7,15
21,CUST023,Rahul Singh,Chennai,5,15
10,CUST011,Meera Iyer,Chennai,6,14
5,CUST005,Aarav Gupta,Mumbai,4,14
27,CUST030,Imran Rao,Kolkata,4,14
18,CUST020,Divya Das,Kochi,3,14
7,CUST007,Lakshmi Rao,Pune,3,12


##### Verify all five views

In [0]:
%sql
SHOW VIEWS IN adlsdatabricks2_workspace.gold;

namespace,viewName,isTemporary,isMaterialized,isMetric
gold,vw_monthly_order_trend,false,false,false
gold,vw_order_status_summary,false,false,false
gold,vw_product_category_summary,false,false,false
gold,vw_sales_channel_summary,false,false,false
gold,vw_top_customers,false,false,false
